<a href="https://colab.research.google.com/github/Binamra00/rs-replication/blob/main/inflection_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚙️ Phase 0: Environment Setup & Data Ingestion
**Objective:** Establish a "One-Click Reproducibility" environment.
This phase dynamically pulls the raw, pre-mined MSR datasets directly from the Zenodo archive. It recursively auto-discovers the extracted JSON/JSONL files (PMD snapshots, RefactoringMiner events, and Lineage histories) and loads them into the active workspace, decoupling the pipeline from any local or Google Drive path dependencies.

In [ ]:
# @title
import os
import urllib.request
import zipfile
from pathlib import Path

# ==========================================
# 🛠️ 1. CONFIGURATION & ZENODO TARGET
# ==========================================
# This ID points to the permanent Zenodo data archive
ZENODO_RECORD_ID = "20617639" # <-- PASTE YOUR ID HERE

# Interactive Colab Dropdown for Reviewers
TARGET_REPO = "commons-lang" # @param ["commons-lang", "checkstyle", "dubbo", "junit4", "questdb"]

print(f"🚀 Setting up replication environment for {TARGET_REPO}...")
!pip install -q pandas scikit-learn packaging numpy

# ==========================================
# 🛠️ 2. WORKSPACE SETUP (Ephemeral / Local)
# ==========================================
BASE_DIR = Path(os.getcwd()).resolve()
DATA_DIR = BASE_DIR / "data" / TARGET_REPO
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Workspace Root: {BASE_DIR}")
print(f"📦 Data Directory: {DATA_DIR}")

# ==========================================
# 🛠️ 3. FETCH PRE-MINED DATA FROM ZENODO
# ==========================================
zip_path = BASE_DIR / f"{TARGET_REPO}.zip"
zenodo_url = f"https://zenodo.org/records/{ZENODO_RECORD_ID}/files/{TARGET_REPO}.zip?download=1"

# Only download if we don't already have the files
if not list(DATA_DIR.rglob("*.jsonl")):
    print(f"☁️ Downloading {TARGET_REPO} data from Zenodo...")
    try:
        urllib.request.urlretrieve(zenodo_url, zip_path)
    except Exception as e:
        raise RuntimeError(f"❌ Failed to download from Zenodo. Check the ZENODO_RECORD_ID. Error: {e}")

    print("🗜️ Unzipping data...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)

    # Cleanup zip file
    zip_path.unlink()
    print("✅ Download and extraction complete.")
else:
    print("✅ Data already exists locally. Skipping download.")

# ==========================================
# 🛠️ 4. AUTO-DISCOVER EXTRACTED FILES
# ==========================================
# Recursively find ALL JSON and JSONL files
all_files = list(DATA_DIR.rglob("*.json*"))

PMD_JSONL_PATH = next((f for f in all_files if "pmd" in f.name.lower()), None)
REFACTORING_JSONL_PATH = next((f for f in all_files if "refactoring" in f.name.lower()), None)
LINEAGE_JSONL_PATH = next((f for f in all_files if "lineage" in f.name.lower()), None)
OFFICIAL_RELEASES_JSON_PATH = next((f for f in all_files if "rel_hist" in f.name.lower()), None)

# TimeSplit Ratio matching the paper (50%)
TIME_SPLIT_RATIO = 0.50

# Output Files (Splits & Results for Phase T.2, T.3, T.4)
T2_TRAIN_CSV_PATH = BASE_DIR / f"t2_train_chrono_50_{TARGET_REPO}.csv"
T2_TEST_CSV_PATH = BASE_DIR / f"t2_test_chrono_50_{TARGET_REPO}.csv"
INFLECTION_RESULTS_PATH = BASE_DIR / f"inflection_thresholds_{TARGET_REPO}.json"

# ==========================================
# 🛠️ 5. VERIFY ENVIRONMENT
# ==========================================
print("\n--- 🎯 Configuration Summary ---")
print(f"Target Repository:  {TARGET_REPO}")
print(f"PMD Data Exists:    {bool(PMD_JSONL_PATH and PMD_JSONL_PATH.exists())}")
print(f"Refm Data Exists:   {bool(REFACTORING_JSONL_PATH and REFACTORING_JSONL_PATH.exists())}")
print(f"Lineage Exists:     {bool(LINEAGE_JSONL_PATH and LINEAGE_JSONL_PATH.exists())}")
print(f"Rel Hist Exists:    {bool(OFFICIAL_RELEASES_JSON_PATH and OFFICIAL_RELEASES_JSON_PATH.exists())}")

if not all([PMD_JSONL_PATH, REFACTORING_JSONL_PATH, LINEAGE_JSONL_PATH, OFFICIAL_RELEASES_JSON_PATH]):
    print("\n⚠️ WARNING: Could not auto-discover all files. Here is what is inside the directory:")
    for f in list(DATA_DIR.rglob("*")):
        print(f" - {f.name}")
    raise FileNotFoundError("Missing required files. Check the Zenodo zip structure.")
else:
    print("✅ Phase 0 Complete! You can now execute Phase 1.1.")

## Phase 1: The Master Lineage & Alias Graphs
Before any machine learning or threshold calibration can occur, we must establish a deterministic "ground truth" timeline. Phase 1 ingests the raw, independent data streams (PMD snapshots, RefactoringMiner events, and the Git ledger) and synchronizes them into a single, cohesive historical graph.

### ⚙️ Phase 1.1: Data Synchronization Check
**Objective:** Verify the structural alignment of the three independent data universes.
**Logic Flow:**
* **Load Ground Truth:** Read the official release history to identify the targeted repository snapshots.
* **Load Lineage:** Read the master Git SHA map to establish the true commit timeline.
* **Verify PMD Snapshots:** Map PMD static analysis snapshots to the timeline, dropping any failed or orphaned analyses that sit outside the true lineage.
* **Verify RefactoringMiner:** Map RefactoringMiner structural events to the timeline to calculate actual refactoring density.
* **Synchronization Report:** Output an integrity report confirming that PMD snapshots and Refactoring events exist on the exact same verified timeline.

In [ ]:

import json
from pathlib import Path

print("🚀 Starting Phase 1.1: Data Synchronization Check")

# ------------------------
# Dependency / input guards
# ------------------------
required_globals = ["LINEAGE_JSONL_PATH", "PMD_JSONL_PATH", "REFACTORING_JSONL_PATH", "OFFICIAL_RELEASES_JSON_PATH"]
for var in required_globals:
    if var not in globals() or globals()[var] is None:
        raise RuntimeError(f"❌ Phase 1.1 Dependency Error: '{var}' is missing. Did you run Phase 0?")

LINEAGE_JSONL_PATH = Path(LINEAGE_JSONL_PATH)
PMD_JSONL_PATH = Path(PMD_JSONL_PATH)
REFACTORING_JSONL_PATH = Path(REFACTORING_JSONL_PATH)
OFFICIAL_RELEASES_JSON_PATH = Path(OFFICIAL_RELEASES_JSON_PATH)

for p in [LINEAGE_JSONL_PATH, PMD_JSONL_PATH, REFACTORING_JSONL_PATH, OFFICIAL_RELEASES_JSON_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"❌ Phase 1.1 Error: Required file not found: {p}")

# 0. Dynamically Calculate Expected Releases
print("📂 Loading Release History Ground Truth...")
try:
    with open(OFFICIAL_RELEASES_JSON_PATH, "r", encoding="utf-8") as f:
        release_data = json.load(f)
        EXPECTED_RELEASE_COUNT = len(release_data)
    print(f"   ✅ Target Releases: {EXPECTED_RELEASE_COUNT} official releases found in configuration.")
except Exception as e:
    raise RuntimeError(f"❌ Failed to parse {OFFICIAL_RELEASES_JSON_PATH.name}. Error: {e}")

# 1. Load Master Lineage SHAs
print("\n📂 Loading Master Lineage (The SHA Map)...")
lineage_shas = set()
lineage_bad_json = 0

with open(LINEAGE_JSONL_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        if not line.strip():
            continue
        try:
            data = json.loads(line)
        except json.JSONDecodeError:
            lineage_bad_json += 1
            continue

        sha = data.get("commit_sha")
        if sha:
            lineage_shas.add(sha)

print(f"   ✅ Lineage Loaded: {len(lineage_shas)} total commits.")
if lineage_bad_json:
    print(f"   ⚠️ Lineage malformed JSON lines skipped: {lineage_bad_json}")

# 2. Check PMD Release Snapshots
print("\n🔍 Step 1: Verifying PMD Snapshots...")
pmd_shas = set()
found_pmd = 0
pmd_bad_json = 0
pmd_null_sha = 0
pmd_outside_lineage = 0
pmd_failed_status = 0

with open(PMD_JSONL_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        if not line.strip():
            continue
        try:
            data = json.loads(line)
        except json.JSONDecodeError:
            pmd_bad_json += 1
            continue

        # Align with B.2 semantics (only successful PMD parses count)
        if data.get("status") != "success":
            pmd_failed_status += 1
            continue

        sha = data.get("sha")
        if not sha:
            pmd_null_sha += 1
            continue

        pmd_shas.add(sha)
        if sha in lineage_shas:
            found_pmd += 1
        else:
            pmd_outside_lineage += 1
            short_sha = sha[:7] if isinstance(sha, str) else str(sha)
            print(f"   ⚠️ Warning: PMD Snapshot {short_sha} is OUTSIDE the lineage!")

# 3. Check Refactoring Miner Coverage
print("\n🔍 Step 2: Verifying Refactoring Miner Ground Truth...")
refm_shas = set()
found_refm = 0
total_refm_entries = 0
refm_bad_json = 0
refm_null_sha = 0

with open(REFACTORING_JSONL_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        if not line.strip():
            continue
        total_refm_entries += 1
        try:
            data = json.loads(line)
        except json.JSONDecodeError:
            refm_bad_json += 1
            continue

        sha = data.get("sha1")  # RefactoringMiner uses 'sha1'
        if not sha:
            refm_null_sha += 1
            continue

        refm_shas.add(sha)
        if sha in lineage_shas:
            found_refm += 1

# 4. Final Synchronization Report
print("-" * 50)
print("📊 DATA SYNCHRONIZATION REPORT")
print(f"   🔹 PMD Snapshots: {found_pmd} valid snapshots verified in lineage (Expected at least: {EXPECTED_RELEASE_COUNT}).")
print(f"   🔹 Refactoring Commits Mapped: {found_refm}")
print(f"   🔹 Refactoring Density: {len(refm_shas)} unique commits contain refactorings.")

# Transparency counters
print("\n🧾 PARSE INTEGRITY")
print(f"   - PMD unique SHAs observed (success only): {len(pmd_shas)}")
print(f"   - PMD malformed lines: {pmd_bad_json}")
print(f"   - PMD failed-status lines skipped: {pmd_failed_status}")
print(f"   - PMD missing sha: {pmd_null_sha}")
print(f"   - PMD outside-lineage SHAs: {pmd_outside_lineage}")
print(f"   - RefMiner total entries read: {total_refm_entries}")
print(f"   - RefMiner malformed lines: {refm_bad_json}")
print(f"   - RefMiner missing sha1: {refm_null_sha}")

if found_pmd >= EXPECTED_RELEASE_COUNT and found_refm > 0:
    print("\n🚀 SUCCESS: The three universes are synchronized!")
    print("   The PMD observation points and Refactoring events exist on the same timeline.")
else:
    print("\n❌ SYNCHRONIZATION FAILURE:")
    if found_pmd < EXPECTED_RELEASE_COUNT:
        print(f"   - Missing PMD releases. Found {found_pmd}, but expected at least {EXPECTED_RELEASE_COUNT}.")
    if found_refm == 0:
        print("   - Zero refactoring commits found in the lineage.")

print("🏁 Phase 1.1 Complete.")

### ⚙️ Phase 1.2: Final Gold Class Alias Graph
**Objective:** Resolve class-level file renames and structural moves over time to preserve canonical identity.
**Logic Flow:**
* **Ingest & Filter:** Load raw RefactoringMiner events and isolate purely structural operations (`Rename Class`, `Move Class`, `Move Source Folder`, `Rename Package`, `Move and Rename Class`).
* **Chronological Sorting:** Sort all extracted events strictly by timestamp to establish an accurate timeline of architectural mutations.
* **Location Extraction:** Extract the "Before" (left side) and "After" (right side) file paths for every structural event.
* **First-Write-Wins Traversal:** Apply a backward-resolving mapping strategy. Traverse the chain of aliases to link every transient name back to its original "root" canonical identity.
* **Artifact Generation:** Export deterministic mapping dictionaries (for both full paths and basenames) to prevent data leakage and secure metric continuity across train/test boundaries.

In [ ]:
# ===============================================
# PHASE 1.2: FINAL GOLD CLASS ALIAS GRAPH (CHRONOLOGICAL, HARDENED)
# ===============================================
import json
import pandas as pd
from pathlib import Path
from collections import Counter

print("🚀 Starting Phase 1.2: Final Gold Class-Level Alias Graph Generation")

# --- Dependency Guards ---
required_globals = ["REFACTORING_JSONL_PATH", "BASE_DIR", "TARGET_REPO"]
for var in required_globals:
    if var not in globals():
        raise RuntimeError(
            f"❌ Phase 1.2 Dependency Error: '{var}' is not defined. "
            "Please set up your environment paths before running."
        )

REFACTORING_JSONL_PATH = Path(REFACTORING_JSONL_PATH)
BASE_DIR = Path(BASE_DIR)
TARGET_REPO = str(TARGET_REPO)

if not REFACTORING_JSONL_PATH.exists():
    raise FileNotFoundError(f"❌ Phase 1.2 Error: Missing input file: {REFACTORING_JSONL_PATH}")

# ---------------------------------------------------------
# Internal identity model (RENAMED to enforce directionality):
# - historical_to_root_path: new_path -> old_path (Legacy backward resolution)
# - historical_to_root_basename: new_basename -> old_basename (Legacy backward resolution)
# ---------------------------------------------------------
historical_to_root_path = {}
historical_to_root_basename = {}

skipped_records = Counter()
collision_count_path = 0
collision_count_basename = 0

def normalize_repo_relative_path(raw_path: str, target_repo: str) -> str:
    """Normalize path to forward slashes, lowercase, and strip any '/<repo>/' prefix."""
    p = (raw_path or "").replace("\\", "/").strip().lower()
    if not p:
        return ""
    needle = f"/{target_repo.lower()}/"
    if needle in p:
        p = p.split(needle, 1)[-1]
    return p.lstrip("/")

# =========================================================
# STEP 1: LOAD AND SORT EVENTS CHRONOLOGICALLY
# =========================================================
raw_rename_events = []

with open(REFACTORING_JSONL_PATH, "r", encoding="utf-8") as f:
    for line_idx, line in enumerate(f):
        if not line.strip():
            continue
        try:
            data = json.loads(line)
        except json.JSONDecodeError:
            skipped_records["Malformed JSON Line"] += 1
            continue

        commit_time = int(data.get("timestamp", data.get("commit_time", 0)) or 0)

        # Use line index as fallback sort key for missing RefMiner timestamps
        sort_key = line_idx if commit_time == 0 else commit_time

        raw_rename_events.append({"time": sort_key, "data": data})

raw_rename_events.sort(key=lambda x: x["time"])
print(f"⏳ Sorted {len(raw_rename_events)} commits chronologically to guarantee safe alias resolution.")

# =========================================================
# STEP 2: BUILD DETERMINISTIC PATH-LEVEL GRAPH
# =========================================================
allowed_refactoring_types = {
    "Rename Class", "Move Class", "Move Source Folder",
    "Rename Package", "Move and Rename Class"
}

for event in raw_rename_events:
    data = event["data"]

    for ref in data.get("refactorings", []):
        if ref.get("type") not in allowed_refactoring_types:
            continue

        left_locs = ref.get("leftSideLocations", [])
        right_locs = ref.get("rightSideLocations", [])

        if not left_locs or not right_locs:
            skipped_records["Missing Location Data"] += 1
            continue

        # Fail loud on asymmetry to prevent fabricating false edges
        if len(left_locs) != len(right_locs):
            skipped_records[f"Asymmetric locations ({ref.get('type')})"] += 1
            continue

        for left_loc, right_loc in zip(left_locs, right_locs):

            old_path_raw = left_loc.get("filePath", "")
            new_path_raw = right_loc.get("filePath", "")

            # --- Legacy Phase 1/2/3 Mapping Logic (Backward: New -> Old) ---
            old_rel = normalize_repo_relative_path(old_path_raw, TARGET_REPO)
            new_rel = normalize_repo_relative_path(new_path_raw, TARGET_REPO)

            if not old_rel or not new_rel:
                skipped_records["Empty Path"] += 1
                continue

            old_basename = old_rel.split("/")[-1]
            new_basename = new_rel.split("/")[-1]

            # First-write-wins to preserve the original historical root
            if old_rel != new_rel:
                if new_rel not in historical_to_root_path:
                    historical_to_root_path[new_rel] = old_rel
                elif historical_to_root_path[new_rel] != old_rel:
                    collision_count_path += 1

            if old_basename != new_basename:
                if new_basename not in historical_to_root_basename:
                    historical_to_root_basename[new_basename] = old_basename
                elif historical_to_root_basename[new_basename] != old_basename:
                    collision_count_basename += 1

print(f"   🔍 Discovered {len(historical_to_root_path)} Class Rename path-level aliases.")
print(f"   🔍 Discovered {len(historical_to_root_basename)} Class Rename basename aliases.")

# --- Canonical Resolvers ---
def resolve_to_historical_root_path(rel_path: str) -> str:
    current = normalize_repo_relative_path(rel_path, TARGET_REPO)
    visited = set()
    while current in historical_to_root_path and current not in visited:
        visited.add(current)
        current = historical_to_root_path[current]
    return current

def resolve_to_historical_root_basename(basename: str) -> str:
    current = (basename or "").strip().lower()
    visited = set()
    while current in historical_to_root_basename and current not in visited:
        visited.add(current)
        current = historical_to_root_basename[current]
    return current

# =========================================================
# STEP 3: SAVE ARTIFACTS
# =========================================================
path_alias_records = []
for new_path in sorted(historical_to_root_path.keys()):
    canonical = resolve_to_historical_root_path(new_path)
    path_alias_records.append({
        "New_Path_Identity": new_path,
        "Canonical_Root_Path_Identity": canonical,
        "New_Basename": new_path.split("/")[-1],
        "Canonical_Root_Basename": canonical.split("/")[-1]
    })

df_path_aliases = pd.DataFrame(path_alias_records)
PATH_ALIAS_CSV_PATH = BASE_DIR / f"alias_tracking_graph_paths_{TARGET_REPO}.csv"
df_path_aliases.to_csv(PATH_ALIAS_CSV_PATH, index=False)

alias_records = [
    {"New_Identity": k, "Canonical_Root_Identity": resolve_to_historical_root_basename(k)}
    for k in sorted(historical_to_root_basename.keys())
]
df_aliases = pd.DataFrame(alias_records)
ALIAS_CSV_PATH = BASE_DIR / f"alias_tracking_graph_{TARGET_REPO}.csv"
df_aliases.to_csv(ALIAS_CSV_PATH, index=False)

# =========================================================
# STEP 4: INTEGRITY REPORT
# =========================================================
print(f"\n📊 PHASE 1.2 INTEGRITY REPORT:")
print(f"   - Total Path Rename Links:      {len(historical_to_root_path)}")
print(f"   - Total Basename Rename Links:  {len(historical_to_root_basename)}")
print(f"   - Path Collisions (Preserved):  {collision_count_path}")
print(f"   - Basename Collisions:          {collision_count_basename}")
print(f"   - Malformed/Skipped:            {sum(skipped_records.values())}")

if skipped_records:
    for reason, count in skipped_records.most_common(5):
        print(f"       * {reason}: {count}")

print(f"\n✅ Identity lineage successfully resolved.")
print(f"   💾 Saved path-level graph to: {PATH_ALIAS_CSV_PATH.name}")
print(f"   💾 Saved basename graph to:   {ALIAS_CSV_PATH.name}")
print("🏁 Phase 1.2 Complete.")

### ⚙️ Phase 1.3: Final Audited Method Alias Graph
**Objective:** Track method-level identity through structural changes and refactoring boundaries.
**Logic Flow:**
* **Class Resolution:** Load the canonical class alias map generated in Phase 1.2 to resolve high-level namespace shifts before tracking inner-method shifts.
* **Operation Filtering:** Isolate `Rename Method` operations from the RefactoringMiner ledger.
* **Strict Hunter Guardrail:** Enforce a strict quality control check: only accept operations that provide a clean `METHOD_DECLARATION` tag to prevent fuzzy signature matching.
* **Signature Normalization:** Apply regex to flatten whitespaces and resolve complex generic types to maintain signature stability over time.
* **Purity Exclusion:** Intentionally omit parameter-change operations to guarantee that the resulting method identity chain remains statistically pure and free from payload artifacts.

In [ ]:
# ===============================================
# PHASE 1.3: FINAL AUDITED METHOD ALIAS GRAPH (HARDENED)
# ===============================================
"""
DESIGN TRADEOFF NOTE (MSR Pipeline):
This phase strictly tracks 'Rename Method' to build the canonical identity chain.
'Change Parameter Type' is explicitly excluded.
REASON: RefactoringMiner currently outputs parameter fragments rather than a reliable
full signature for parameter-change events in many cases.
CONSEQUENCE: Identity chains may break across parameter-change boundaries.
This is an accepted tradeoff for high-confidence rename extraction.
"""
import json
import pandas as pd
import re
from pathlib import Path
from collections import namedtuple, Counter

print("🚀 Starting Phase 1.3: Final Audited Method Alias Graph Generation")

# --- Dependency Guards ---
# BUG 1 FIX: Updated dependency guard to use the correct function name from Phase 1.2
required_globals = ["REFACTORING_JSONL_PATH", "BASE_DIR", "TARGET_REPO", "resolve_to_historical_root_basename"]
for var in required_globals:
    if var not in globals():
        raise RuntimeError(f"❌ Phase 1.3 Dependency Error: '{var}' is missing.")

REFACTORING_JSONL_PATH = Path(REFACTORING_JSONL_PATH)
BASE_DIR = Path(BASE_DIR)
TARGET_REPO = str(TARGET_REPO)

if not REFACTORING_JSONL_PATH.exists():
    raise FileNotFoundError(f"❌ Phase 1.3 Error: Missing input file: {REFACTORING_JSONL_PATH}")

MethodID = namedtuple("MethodID", ["file", "sig"])

method_alias_map = {}
# BUG 2 FIX: Use Counters for tracking to prevent memory bloat
unmatched_elements = Counter()
skipped_records = Counter()
collision_count = 0
zero_timestamp_events = 0

EXCLUDED_TYPES = {
    "Change Parameter Type": "Excluded by design tradeoff (signature instability in source payload)"
}

# NIT 1 FIX: Confirmed the regex correctly handles '$' for inner classes
signature_regex = re.compile(r'([A-Za-z_$][\w$]*)\s*\((.*)\)')

def _strip_param_annotations_and_modifiers(param: str) -> str:
    """
    Normalize a single parameter fragment by removing common Java modifiers/annotations
    while preserving type shape as much as possible.
    """
    p = (param or "").strip()
    if not p:
        return p

    # BUG 4 FIX: More robust annotation stripping, handles simple nested parens and adds missing Java modifiers
    p = re.sub(r'@\w+(\([^)]*\))?\s*', '', p)
    # Added static, public, private, protected, synchronized to the strip list
    p = re.sub(r'\b(final|volatile|transient|static|public|private|protected|synchronized)\b\s*', '', p)

    # Normalize varargs spacing: "String ... args" -> "String... args"
    p = re.sub(r'\s*\.\.\.\s*', '... ', p).strip()

    return p

def extract_signature(code_element: str):
    """
    Extract normalized 'name(type1,type2,...)' signature from a codeElement-like string.
    Handles generic commas by depth tracking.
    """
    if not code_element:
        return None

    match = signature_regex.search(code_element)
    if not match:
        return None

    method_name = match.group(1)
    raw_params = match.group(2)

    params_list = []
    depth = 0
    current = ""

    for ch in raw_params:
        if ch == '<':
            depth += 1
        elif ch == '>':
            depth = max(0, depth - 1)

        if ch == ',' and depth == 0:
            params_list.append(current.strip())
            current = ""
        else:
            current += ch
    if current.strip():
        params_list.append(current.strip())

    normalized_types = []
    for p in params_list:
        p = _strip_param_annotations_and_modifiers(p)

        # split last token as presumed variable name, keep type side
        parts = p.rsplit(" ", 1)
        candidate = parts[0] if len(parts) > 1 else p
        candidate = candidate.strip()

        # BUG 5 COMMENT: Assumption -> Legacy C-style array declarations (e.g. "String args[]")
        # will lose their brackets during this split. This is acceptable given their rarity in modern Java.

        if candidate:
            normalized_types.append(candidate)

    return f"{method_name}({','.join(normalized_types)})"

# =========================================================
# STEP 1: LOAD & SORT EVENTS CHRONOLOGICALLY
# =========================================================
raw_events = []

with open(REFACTORING_JSONL_PATH, "r", encoding="utf-8") as f:
    for line_idx, line in enumerate(f):
        if not line.strip():
            continue
        try:
            data = json.loads(line)
        except json.JSONDecodeError:
            skipped_records["Malformed JSON Line"] += 1
            continue

        commit_time = int(data.get("timestamp", data.get("commit_time", 0)) or 0)

        # BUG 2 FIX: Same logic applied as in 1.2 to safely handle missing timestamps
        if commit_time == 0:
            zero_timestamp_events += 1
            sort_key = line_idx
        else:
            sort_key = commit_time

        raw_events.append({"time": sort_key, "data": data})

raw_events.sort(key=lambda x: x["time"])
print(f"⏳ Sorted {len(raw_events)} commits chronologically for deterministic method alias chaining.")

# =========================================================
# STEP 2: BUILD DETERMINISTIC METHOD GRAPH
# =========================================================
for event in raw_events:
    data = event["data"]

    for ref in data.get("refactorings", []):
        ref_type = ref.get("type")

        if ref_type in EXCLUDED_TYPES:
            skipped_records[ref_type] += 1
            continue

        if ref_type != "Rename Method":
            continue

        left_locs = ref.get("leftSideLocations", [])
        right_locs = ref.get("rightSideLocations", [])

        if not left_locs or not right_locs:
            skipped_records["Rename Method missing locations"] += 1
            continue

        # Fail loud on asymmetry to prevent fabricating false method edges
        if len(left_locs) != len(right_locs):
            skipped_records["Rename Method asymmetric locations"] += 1
            continue

        for left, right in zip(left_locs, right_locs):

            old_file_raw = left.get("filePath", "")
            if not old_file_raw:
                skipped_records["Rename Method missing filePath"] += 1
                continue

            old_file_basename = old_file_raw.replace("\\", "/").lower().split("/")[-1]

            # BUG 1 FIX: Using the corrected resolver function name
            canonical_file = resolve_to_historical_root_basename(old_file_basename)

            code_left = left.get("codeElement", "")
            code_right = right.get("codeElement", "")

            # BUG 3 FIX: Explicitly track empty codeElements to distinguish from parse failures
            if not code_left:
                skipped_records["Rename Method empty codeElement (left)"] += 1
                continue
            if not code_right:
                skipped_records["Rename Method empty codeElement (right)"] += 1
                continue

            old_sig = extract_signature(code_left)
            new_sig = extract_signature(code_right)

            if not old_sig:
                unmatched_elements[code_left] += 1
            if not new_sig:
                unmatched_elements[code_right] += 1

            if not (canonical_file and old_sig and new_sig):
                skipped_records["Rename Method unresolved signature"] += 1
                continue

            if old_sig == new_sig:
                skipped_records["Rename Method no-op after normalization"] += 1
                continue

            child = MethodID(canonical_file, new_sig)
            parent = MethodID(canonical_file, old_sig)

            # BUG 2 FIX: Implemented First-Write-Wins policy to remain consistent with Phase 1.2
            if child not in method_alias_map:
                method_alias_map[child] = parent
            elif method_alias_map[child] != parent:
                collision_count += 1

print(f"   🔍 Discovered {len(method_alias_map)} method lineage links.")

# =========================================================
# STEP 3: CANONICAL RESOLVER
# =========================================================
# BUG 6 COMMENT: The D.2 logic MUST invoke this function per-key, not manually walk the map.
def get_canonical_method_signature(canonical_file: str, current_sig: str) -> str:
    identity = MethodID(canonical_file, current_sig)
    visited = set()
    while identity in method_alias_map and identity not in visited:
        visited.add(identity)
        identity = method_alias_map[identity]
    return identity.sig

# =========================================================
# STEP 4: SERIALIZE & SAVE
# =========================================================
method_alias_records = [
    {
        "Canonical_File": k.file,
        "New_Signature": k.sig,
        "Canonical_Root_Signature": get_canonical_method_signature(k.file, k.sig)
    }
    for k in sorted(method_alias_map.keys(), key=lambda x: (x.file, x.sig))
]

df_method_aliases = pd.DataFrame(method_alias_records)
METHOD_ALIAS_CSV_PATH = BASE_DIR / f"method_alias_graph_{TARGET_REPO}.csv"
df_method_aliases.to_csv(METHOD_ALIAS_CSV_PATH, index=False)

print("   ✅ Method lineage resolved.")
print(f"   💾 Saved to {METHOD_ALIAS_CSV_PATH.name}")

# =========================================================
# STEP 5: INTEGRITY REPORT
# =========================================================
print("\n📊 PHASE 1.3 INTEGRITY REPORT:")
print(f"   - Total Method Links Mapped: {len(method_alias_map)}")
print(f"   - Collision Overwrites:      {collision_count}")

if skipped_records:
    print("   - Explicitly Skipped / Excluded:")
    for k, v in sorted(skipped_records.items(), key=lambda x: x[0]):
        print(f"       * {k}: {v}")

if unmatched_elements:
    # NIT 2 FIX: Improved diagnostic reporting for regex failures
    print(f"\n   ⚠️ Regex Extraction Failures (unique patterns): {len(unmatched_elements)}")
    print(f"   ⚠️ Regex Extraction Failures (total events): {sum(unmatched_elements.values())}")
    print(f"   - Top failures:")
    for pattern, count in unmatched_elements.most_common(5):
        print(f"       * {count}x {pattern[:80]}")
else:
    print("\n   ✅ DATA INTEGRITY VERIFIED: 100% Regex match rate on targeted elements.")

print("\n🏁 Phase 1.3 Complete.")

## Phase 2: Matrix Engineering & Multi-Regime Partitioning
This phase fuses the static analysis metrics with the refactoring ground truth to build the foundational Machine Learning matrices. It then partitions the data into three distinct evaluation tracks: Spatial (Traditional), Scarcity (Volume Stress-Test), and Chronological (Time-Travel Defense).

### ⚙️ Phase D.2: Data Flattening & Early Binding
**Objective:** Construct the foundational Tabular Matrix mapping structural smells to refactoring ground truth.
**Logic Flow:**
* **Data Ingestion:** Load the raw PMD static analysis snapshots and the parsed RefactoringMiner event ledger.
* **Identity Unification:** Apply the alias graphs from Phases 1.2 and 1.3 to normalize target entities across both streams.
* **Taxonomy Enforcement:** Execute a strict *Smell-to-Refactoring Taxonomy*. PMD warnings are bound exclusively to logically corresponding refactorings (e.g., God Class metrics are supervised *only* by Class Extraction/Moves).
* **Metric Engineering:** Calculate aggregated file-level metrics (such as `Calculated_MethodCount`) to provide continuous variables for whole-class rules like `TooManyMethods`.
* **Matrix Flattening:** Compile the cross-referenced streams into a single Tabular CSV, firmly tagging positive (`is_refactored=1`) and negative (`is_refactored=0`) observations.

In [ ]:
# ==========================================
# PHASE D.2: DATA FLATTENING & EARLY BINDING (HARDENED)
# ==========================================
import re
import json
import bisect
import pandas as pd
from pathlib import Path
from collections import Counter

print("🚀 Starting Phase D.2: Data Flattening & Regex Rescue (With Alias Resolution)")

# ------------------------
# DEPENDENCY / INPUT GUARDS
# ------------------------
required_globals = [
    "TARGET_REPO", "BASE_DIR",
    "LINEAGE_JSONL_PATH", "PMD_JSONL_PATH", "REFACTORING_JSONL_PATH",
    "resolve_to_historical_root_basename", "get_canonical_method_signature", "extract_signature", "method_alias_map"
]
for var in required_globals:
    if var not in globals() or globals()[var] is None:
        raise RuntimeError(f"❌ Phase D.2 Dependency Error: '{var}' is missing or None. Did you run previous phases?")

base_dir_p = Path(BASE_DIR)
lineage_p = Path(LINEAGE_JSONL_PATH)
pmd_p = Path(PMD_JSONL_PATH)
refactoring_p = Path(REFACTORING_JSONL_PATH)

flat_pmd_csv_p = base_dir_p / f"pmd_flat_{TARGET_REPO}.csv"

for p in [lineage_p, pmd_p, refactoring_p]:
    if not p.exists():
        raise FileNotFoundError(f"❌ Phase D.2 Error: Required input file not found: {p}")

print("⏳ Building PMD Method-Name Resolver from Phase 1.3 Graph...")
pmd_method_resolver = {}
resolver_collisions = 0

# Ensure D.2 uses the resolver properly, not a manual walk
for method_id in sorted(method_alias_map.keys(), key=lambda x: str(x)):
    file_box = method_id.file
    current_name_only = method_id.sig.split("(")[0]

    # This properly leverages the cycle-safe while loop from 1.3
    root_sig = get_canonical_method_signature(file_box, method_id.sig)
    root_name_only = root_sig.split("(")[0]

    key = f"{file_box}::{current_name_only}"
    prev = pmd_method_resolver.get(key)

    # First-Write-Wins logic (matching 1.2 and 1.3)
    if prev is None:
        pmd_method_resolver[key] = root_name_only
    elif prev != root_name_only:
        resolver_collisions += 1

if resolver_collisions:
    print(f"   ⚠️ Resolver collisions (preserved original root): {resolver_collisions}")

TEST_FILE_REGEX = re.compile(r'(^Test.*|.*Test(?:s|Case)?)\.java$')

def is_test_artifact(norm_path: str, raw_basename: str) -> bool:
    return ("/test/" in norm_path) or bool(TEST_FILE_REGEX.search(raw_basename or ""))

def normalize_path(path: str) -> str:
    return (path or "").replace("\\", "/").lower()

def normalize_repo_relative_path(path: str, target_repo: str) -> str:
    p = normalize_path(path)
    needle = f"/{target_repo.lower()}/"
    if needle in p:
        p = p.split(needle, 1)[-1]
    return p.lstrip("/")

def rescue_metric_score(rule, desc, fallback_score):
    if fallback_score is not None:
        try:
            fs = float(fallback_score)
            if fs > 0:
                return int(fs)
        except (TypeError, ValueError):
            pass
    desc = str(desc or "")
    try:
        if "ExcessiveImports" in rule:
            m = re.search(r'imports? \((\d+)\)', desc, re.I); return int(m.group(1)) if m else None
        elif "CouplingBetweenObjects" in rule:
            m = re.search(r'value of (\d+)', desc, re.I); return int(m.group(1)) if m else None
        elif "ExcessivePublicCount" in rule:
            m = re.search(r'has (\d+) public', desc, re.I); return int(m.group(1)) if m else None
        elif "ExcessiveParameterList" in rule:
            m = re.search(r'\((\d+) parameter', desc, re.I); return int(m.group(1)) if m else None
        elif "TooManyFields" in rule:
            m = re.search(r'has (\d+) fields?', desc, re.I); return int(m.group(1)) if m else None
        elif "TooManyMethods" in rule:
            m = re.search(r'has (\d+) methods?', desc, re.I); return int(m.group(1)) if m else None
        elif rule in ["NcssCount", "CyclomaticComplexity"]:
            return None
        else:
            return None
    except Exception:
        return None

print("⏳ Loading Timelines and Mapping Ground Truth (Early Binding)...")
commit_times = {}
lineage_bad_json = 0

with open(lineage_p, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip(): continue
        try:
            data = json.loads(line)
            sha = data.get("commit_sha")
            if sha:
                commit_times[sha] = int(data.get("timestamp", 0) or 0)
        except Exception:
            lineage_bad_json += 1

if lineage_bad_json:
    print(f"   ⚠️ Lineage malformed/invalid lines skipped: {lineage_bad_json}")

pmd_release_shas = set()
pmd_bad_json = 0
pmd_failed_status = 0

with open(pmd_p, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip(): continue
        try:
            data = json.loads(line)
        except json.JSONDecodeError:
            pmd_bad_json += 1
            continue
        if data.get("status") != "success":
            pmd_failed_status += 1
            continue
        sha = data.get("sha")
        if sha is not None:
            pmd_release_shas.add(sha)

release_shas = sorted([sha for sha in pmd_release_shas if commit_times.get(sha, 0) > 0], key=lambda x: (commit_times[x], x))
release_times_sorted = [commit_times[sha] for sha in release_shas]

def map_to_preceding_pmd_sha(ref_time: int):
    idx = bisect.bisect_left(release_times_sorted, ref_time) - 1
    return release_shas[idx] if idx >= 0 else None

# ──────────────────────────────────────────────────────────────────────
# TYPE-SPECIFIC SMELL→REFACTORING TAXONOMY (construct-validity fix)
# Each PMD metric is supervised ONLY by refactorings that mechanically
# repay its specific smell — no cross-smell contamination.
# ──────────────────────────────────────────────────────────────────────
SMELL_TO_REFTYPES = {
    "GodClass":    {"Extract Class", "Move Class", "Extract Superclass",
                    "Extract Interface", "Extract Subclass", "Move And Rename Class"},
    "LongMethod":  {"Extract Method", "Inline Method", "Remove Parameter"},
    "FeatureEnvy": {"Move Method", "Pull Up Method", "Push Down Method",
                    "Extract And Move Method"},
}
REFTYPE_TO_SMELL = {rt: s for s, rts in SMELL_TO_REFTYPES.items() for rt in rts}

CLASS_LEVEL_REFS = SMELL_TO_REFTYPES["GodClass"]
METHOD_LEVEL_REFS = SMELL_TO_REFTYPES["LongMethod"] | SMELL_TO_REFTYPES["FeatureEnvy"]

master_ref_events = {
    "GodClass":    set(),   # keys: (sha, basename)
    "LongMethod":  set(),   # keys: (sha, basename, method_name)
    "FeatureEnvy": set(),   # keys: (sha, basename)  [method-move aggregated to class]
}

RULE_TO_SMELL = {
    "NcssCount_Class":             "GodClass",
    "CyclomaticComplexity_Class":  "GodClass",
    "TooManyMethods":              "GodClass",
    "ExcessivePublicCount":        "GodClass",
    "NcssCount_Method":            "LongMethod",
    "CyclomaticComplexity_Method": "LongMethod",
    "CouplingBetweenObjects":      "FeatureEnvy",
    "ExcessiveImports":            "FeatureEnvy",
}

EXCLUDED_RULES = {"ExcessiveParameterList", "TooManyFields"}

raw_taxonomy_matched_refactorings = 0
total_leftside_locations = 0
total_structural_candidates = 0
production_candidates = 0
refm_bad_json = 0
mapped_key_hits_total = 0

ref_sha_missing_from_lineage = 0
unparseable_method_refs = 0

with open(refactoring_p, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip(): continue
        try:
            data = json.loads(line)
        except json.JSONDecodeError:
            refm_bad_json += 1
            continue

        ref_sha = data.get("sha1")

        ref_time = commit_times.get(ref_sha, 0)
        if ref_time == 0:
            ref_sha_missing_from_lineage += 1
            continue
        mapped_pmd_sha = map_to_preceding_pmd_sha(ref_time)

        for ref in data.get("refactorings", []):
            ref_type = ref.get("type")
            if ref_type not in CLASS_LEVEL_REFS and ref_type not in METHOD_LEVEL_REFS:
                continue
            smell = REFTYPE_TO_SMELL.get(ref_type)

            raw_taxonomy_matched_refactorings += 1
            left_locations = ref.get("leftSideLocations", [])
            total_leftside_locations += len(left_locations)

            if ref_type in METHOD_LEVEL_REFS:
                locations_to_process = [
                    loc for loc in left_locations
                    if loc.get("codeElementType") == "METHOD_DECLARATION"
                ]
                if not locations_to_process:
                    locations_to_process = [
                        loc for loc in left_locations
                        if loc.get("codeElement")
                    ]
            else:
                locations_to_process = left_locations

            for loc in locations_to_process:
                total_structural_candidates += 1

                original_path = loc.get("filePath", "").replace("\\", "/")
                norm_path = normalize_repo_relative_path(original_path, TARGET_REPO)
                raw_basename = original_path.split("/")[-1] if original_path else ""

                if is_test_artifact(norm_path, raw_basename):
                    continue

                production_candidates += 1
                canonical_basename = resolve_to_historical_root_basename(raw_basename.lower())

                if smell in ("GodClass", "FeatureEnvy"):
                    if mapped_pmd_sha:
                        mapped_key_hits_total += 1
                        master_ref_events[smell].add((mapped_pmd_sha, canonical_basename))
                    continue

                code_el = str(loc.get("codeElement", ""))
                raw_sig = extract_signature(code_el)
                if raw_sig:
                    canonical_sig = get_canonical_method_signature(canonical_basename, raw_sig)
                    m_name = canonical_sig.split("(")[0]
                else:
                    m_match = re.search(r'(\w+)\s*\(', code_el)
                    if m_match:
                        m_name = m_match.group(1)
                        m_name = pmd_method_resolver.get(f"{canonical_basename}::{m_name}", m_name)
                    else:
                        unparseable_method_refs += 1
                        if unparseable_method_refs <= 10:
                            print(f"   [DEBUG] GENUINE failure on METHOD_DECLARATION: {repr(code_el[:200])}")
                        continue

                if mapped_pmd_sha:
                    mapped_key_hits_total += 1
                    master_ref_events["LongMethod"].add((mapped_pmd_sha, canonical_basename, m_name))

total_supervision_keys = sum(len(s) for s in master_ref_events.values())

print("\n📉 THE DATA ATTRITION FUNNEL (Ground Truth Mapping):")
print(f"   🧩 Raw Taxonomy-Matched Refactorings: {raw_taxonomy_matched_refactorings}")
print(f"   📌 Total leftSideLocations (all matched refs): {total_leftside_locations}")
print(f"   🔍 Total Structural Operations Found: {total_structural_candidates}")
print(f"   🛡️  Test Files Purged (Regex Filter):  {total_structural_candidates - production_candidates}")
print(f"   ✅ Production Refactorings Retained (location-level):  {production_candidates}")
print(f"   🎯 Mappable retained locations (pre-dedup): {mapped_key_hits_total}")
print(f"   🔗 Unique mapped supervision keys (post-dedup): {total_supervision_keys}")
print(f"        ├─ God Class    (class-level):      {len(master_ref_events['GodClass'])}")
print(f"        ├─ Long Method  (method-level):     {len(master_ref_events['LongMethod'])}")
print(f"        └─ Feature Envy (class-aggregated): {len(master_ref_events['FeatureEnvy'])}")
print(f"   ♻️  Dedup/compression delta: {mapped_key_hits_total - total_supervision_keys}")

if ref_sha_missing_from_lineage:
    print(f"   ⚠️ Refactorings dropped (missing from lineage gap): {ref_sha_missing_from_lineage}")
if unparseable_method_refs:
    print(f"   ⚠️ Method refactorings dropped (unparseable signature): {unparseable_method_refs}")

if refm_bad_json:
    print(f"   ⚠️ Refactoring malformed JSON lines:   {refm_bad_json}")
if pmd_bad_json or pmd_failed_status:
    print(f"   ⚠️ PMD malformed lines:                {pmd_bad_json}")
    print(f"   ⚠️ PMD failed-status entries ignored:  {pmd_failed_status}")
print()

records = []
pmd_parse_bad_json = 0
unrecoverable_metric_rows = 0
excluded_rule_rows = {r: 0 for r in EXCLUDED_RULES}

with open(pmd_p, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip(): continue
        try:
            commit_data = json.loads(line)
        except json.JSONDecodeError:
            pmd_parse_bad_json += 1
            continue

        if commit_data.get("status") != "success": continue
        sha = commit_data.get("sha")
        if not sha: continue

        for file_obj in commit_data.get("violations", []):
            original_path = file_obj.get("filename", "").replace("\\", "/")
            norm_path = normalize_repo_relative_path(original_path, TARGET_REPO)
            raw_basename = original_path.split("/")[-1] if original_path else ""

            if is_test_artifact(norm_path, raw_basename): continue

            canonical_basename = resolve_to_historical_root_basename(raw_basename.lower())

            for violation in file_obj.get("violations", []):
                rule = violation.get("rule", "")

                if rule in EXCLUDED_RULES:
                    excluded_rule_rows[rule] += 1
                    continue

                raw_score = violation.get("metric_value", 0)
                desc = violation.get("description", "")
                real_score = rescue_metric_score(rule, desc, raw_score)

                if real_score is None:
                    unrecoverable_metric_rows += 1
                    continue

                method_name = violation.get("method") or "class_level"

                if method_name == "class_level" and ("method '" in desc or "constructor '" in desc):
                    m = re.search(r"(?:method|constructor) '([^']+)'", desc)
                    if m:
                        method_name = m.group(1).split("(")[0]

                if method_name != "class_level":
                    method_name = pmd_method_resolver.get(f"{canonical_basename}::{method_name}", method_name)

                if rule in ["NcssCount", "CyclomaticComplexity"]:
                    suffix = "Class" if method_name == "class_level" else "Method"
                    granular_rule = f"{rule}_{suffix}"
                else:
                    granular_rule = rule

                rule_smell = RULE_TO_SMELL.get(granular_rule)
                if rule_smell == "LongMethod":
                    is_ref = (sha, canonical_basename, method_name) in master_ref_events["LongMethod"]
                elif rule_smell in ("GodClass", "FeatureEnvy"):
                    is_ref = (sha, canonical_basename) in master_ref_events[rule_smell]
                else:
                    is_ref = False

                records.append({
                    "commit_sha": sha,
                    "file_path": norm_path,
                    "raw_basename": raw_basename,
                    "file_basename": canonical_basename,
                    "rule": rule,
                    "granular_rule": granular_rule,
                    "metric_value": int(real_score),
                    "method_name": method_name,
                    "is_refactored": 1 if is_ref else 0
                })

df_pmd = pd.DataFrame(records)
if df_pmd.empty:
    raise RuntimeError("❌ Phase D.2 Fatal Error: Flattened PMD dataset is empty. Cannot continue.")

# ── Engineered metric: per-class method count (for TooManyMethods) ──
_mc = (df_pmd[df_pmd["method_name"] != "class_level"]
       .groupby(["commit_sha", "file_basename"])["method_name"]
       .nunique().rename("Calculated_MethodCount").reset_index())

df_pmd = df_pmd.merge(_mc, on=["commit_sha", "file_basename"], how="left")
df_pmd["Calculated_MethodCount"] = df_pmd["Calculated_MethodCount"].fillna(0).astype(int)

print(f"✅ Flattened PMD Data: {len(df_pmd):,} raw metric records extracted.")

print("\n🚫 Excluded-rule rows dropped from dataset (reported as a finding):")
for r in sorted(excluded_rule_rows):
    reason = ("no method signature → unmappable" if r == "ExcessiveParameterList"
              else "no metric value/range emitted" if r == "TooManyFields"
              else "excluded")
    print(f"      - {r:<28}: {excluded_rule_rows[r]:>8,} rows  ({reason})")

# =====================================================================
# --- GLOBAL ZERO REFACTORING DIAGNOSTIC (WITH RELEASE NUMBERS) ---
# =====================================================================
print("\n🔍 DIAGNOSTIC: Checking for Zero-Refactoring Releases across the entire timeline...")

sha_to_release_num = {sha: idx + 1 for idx, sha in enumerate(release_shas)}
total_releases = len(release_shas)

zero_ref_releases = []
for sha, group in df_pmd.groupby("commit_sha"):
    if group["is_refactored"].sum() == 0:
        zero_ref_releases.append(sha)

if zero_ref_releases:
    print(f"   ⚠️ WARNING: Found {len(zero_ref_releases)} releases with ZERO mapped refactoring events.")
    print("   (These releases will likely be dropped during downstream Top-K or F-score evaluations).")

    zero_ref_releases_sorted = sorted(zero_ref_releases, key=lambda x: sha_to_release_num.get(x, 999))

    for i, sha in enumerate(zero_ref_releases_sorted):
        rel_num = sha_to_release_num.get(sha, 'Unknown')
        print(f"      - Release {rel_num}/{total_releases} : {sha}")
else:
    print("   ✅ All releases contain at least one mapped refactoring event.")
print("-" * 85 + "\n")

if unrecoverable_metric_rows:
    print(f"   ⚠️ Dropped unrecoverable metric rows: {unrecoverable_metric_rows}")
if pmd_parse_bad_json:
    print(f"   ⚠️ Additional malformed PMD lines skipped during flatten: {pmd_parse_bad_json}")

df_pmd.to_csv(flat_pmd_csv_p, index=False)
print(f"\n🏁 Phase D.2 Complete! Saved flat dataset to {flat_pmd_csv_p.name}")
print("Note: Train/Test splitting happens in Phase T.2.")

#### 🔎 Phase D.2 Audit: Post-Flattening Integrity Check
**Objective:** Validate the structural integrity, class imbalance, and metric completeness of the final flattened matrix.
**Logic Flow:**
* **Data Ingestion:** Load the newly generated D.2 Tabular Matrix.
* **Sparsity Quantification:** Calculate the exact positive/negative class imbalance ratio to formally document the "Data Scarcity" baseline for the dataset.
* **Null-Value Sweep:** Execute a strict diagnostic to verify `Missing/Null Metric Values` is exactly 0, proving the upstream PMD parser successfully extracted structural values without data corruption.
* **Coverage Verification:** Output row counts segmented by rule type to ensure consistent application of the `is_test_artifact` filters and correct methodological coverage across all tracked metrics.

In [ ]:
print("\n📊 POST-FLATTENING DATASET AUDIT:")

try:
    df_audit = pd.read_csv(flat_pmd_csv_p)

    # 1. Basic Stats
    total_rows = len(df_audit)
    total_positive = df_audit["is_refactored"].sum()
    total_negative = total_rows - total_positive
    imbalance_ratio = total_negative / total_positive if total_positive > 0 else 0

    # 2. Integrity Stats
    null_metrics = df_audit["metric_value"].isnull().sum()
    unique_files = df_audit["file_basename"].nunique()
    unique_commits = df_audit["commit_sha"].nunique()

    print(f"   - Total Observations: {total_rows:,}")
    print(f"   - Positive Refactoring Labels: {total_positive:,}")
    print(f"   - Negative Observations: {total_negative:,}")
    print(f"   - Class Imbalance Ratio: {imbalance_ratio:.2f}:1")
    print(f"   - Unique Files Tracked: {unique_files:,}")
    print(f"   - Unique Commits (Snapshots) Tracked: {unique_commits:,}")
    print(f"   - Missing/Null Metric Values: {null_metrics:,}")

    print("\n   - Row Count by Rule Type:")
    rule_counts = df_audit["granular_rule"].value_counts()
    for rule, count in rule_counts.items():
        print(f"       * {rule:<25}: {count:,}")

    if total_positive == 0:
        print("\n⚠️ WARNING: The dataset contains ZERO refactoring events. Check alignment between lineage and refminer.")
    else:
        print("\n✅ Dataset audit passed: Structure is valid and data is balanced.")

except Exception as e:
    print(f"   ⚠️ Could not audit dataset: {e}")

#### 🧪 Phase D.2 Audit: Boiling-Frog Hypothesis Test (Exploratory Analysis)
**Objective:** Empirically determine if metrics follow a 1D monotonic threshold or a 2D peaked range model.
**Logic Flow:**
* **Quantile Binning:** Segment the metric values for each PMD rule into equal-frequency quantile bins. This smooths out the noisy, sparse tails of high-metric values.
* **Empirical Rate Calculation:** Compute the empirical refactoring rate $P(refactored | metric\_value = v)$ for each bin, applying Wilson 95% confidence intervals to quantify per-bin uncertainty.
* **Automated Classification:** Calculate the Spearman rank correlation and peak-drop percentages across the bins.
* **Behavioral Profiling:** Automatically classify the rule's behavioral profile as either **Monotonic** (the classic assumption: worse smells are fixed more often) or **Peaked** (the Boiling Frog effect: refactoring likelihood drops once a class becomes too toxic/complex to safely touch).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

# -----------------------------------------------------------------------------
# DYNAMIC CONFIG (Auto-detected from Phase D.2)
# -----------------------------------------------------------------------------
# Verify variables exist from Phase D.2
if "flat_pmd_csv_p" not in globals():
    flat_pmd_csv_p = Path(BASE_DIR) / f"pmd_flat_{TARGET_REPO}.csv"

FLAT_PMD_CSV = flat_pmd_csv_p
REPO_NAME = TARGET_REPO

# Rules to plot
CLASS_LEVEL_RULES = [
    "NcssCount_Class", "CyclomaticComplexity_Class", "ExcessivePublicCount",
    "CouplingBetweenObjects", "ExcessiveImports", "TooManyMethods",
]
METHOD_LEVEL_RULES = [
    "NcssCount_Method", "CyclomaticComplexity_Method", "ExcessiveParameterList",
]

N_BINS = 12
MIN_POSITIVE_PER_BIN = 3

# -----------------------------------------------------------------------------
# Analysis Logic
# -----------------------------------------------------------------------------
def wilson_ci(k, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    p_hat = k / n
    denom = 1 + z**2 / n
    centre = (p_hat + z**2 / (2 * n)) / denom
    margin = (z * np.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2))) / denom
    return (max(0.0, centre - margin), min(1.0, centre + margin))

def analyze_rule(df_rule, rule_name, ax):
    df_rule = df_rule[df_rule["metric_value"] > 0].copy()
    n_total = len(df_rule)
    n_positive = int(df_rule["is_refactored"].sum())

    if n_total < 50 or n_positive < 10:
        ax.set_title(f"{rule_name}\n[insufficient]", fontsize=9, color="gray")
        ax.axis("off")
        return "insufficient", None

    try:
        df_rule["bin"] = pd.qcut(df_rule["metric_value"], q=N_BINS, duplicates="drop", labels=False)
    except:
        ax.axis("off"); return "insufficient", None

    grouped = df_rule.groupby("bin").agg(
        bin_median=("metric_value", "median"),
        bin_n=("is_refactored", "size"),
        bin_pos=("is_refactored", "sum"),
    ).reset_index()

    grouped["rate"] = grouped["bin_pos"] / grouped["bin_n"]
    grouped[["ci_lo", "ci_hi"]] = grouped.apply(lambda r: pd.Series(wilson_ci(r["bin_pos"], r["bin_n"])), axis=1)

    plotted = grouped[grouped["bin_pos"] >= MIN_POSITIVE_PER_BIN]
    if len(plotted) < 4:
        ax.axis("off"); return "insufficient", None

    ax.errorbar(plotted["bin_median"], plotted["rate"],
                yerr=[plotted["rate"] - plotted["ci_lo"], plotted["ci_hi"] - plotted["rate"]],
                fmt="o-", capsize=3, linewidth=1.5, markersize=5, color="#1f77b4")

    peak_idx = plotted["rate"].idxmax()
    ax.axvline(plotted.loc[peak_idx, "bin_median"], color="red", linestyle="--", alpha=0.5, linewidth=1)

    rho, p_val = stats.spearmanr(plotted["bin_median"], plotted["rate"])
    last_rate, peak_rate, first_rate = plotted["rate"].iloc[-1], plotted["rate"].max(), plotted["rate"].iloc[0]

    verdict = "ambiguous"
    if rho > 0.7 and p_val < 0.05: verdict = "monotonic_up"
    elif rho < -0.7 and p_val < 0.05: verdict = "monotonic_down"
    elif (peak_rate - last_rate)/peak_rate >= 0.25 and (peak_rate - first_rate)/peak_rate >= 0.25: verdict = "peaked"
    elif abs(rho) < 0.3: verdict = "flat"

    ax.set_title(f"{rule_name}\n[{verdict}]", fontsize=9)

    # ADDED: Explicit Axis Labels
    ax.set_xlabel("Metric Value (Bin Median)", fontsize=8)
    ax.set_ylabel("P(Refactored)", fontsize=8)

    ax.grid(alpha=0.3)
    return verdict, {"rule": rule_name, "verdict": verdict, "peak_rate": peak_rate}

# -----------------------------------------------------------------------------
# EXECUTION
# -----------------------------------------------------------------------------
df = pd.read_csv(FLAT_PMD_CSV)
print(f"Audit loaded: {len(df):,} records for {REPO_NAME}")

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
all_rules = CLASS_LEVEL_RULES + METHOD_LEVEL_RULES
results = []
for i, rule in enumerate(all_rules):
    verdict, stats_row = analyze_rule(df[df["granular_rule"] == rule], rule, axes.flatten()[i])
    if stats_row: results.append(stats_row)

plt.tight_layout()
plt.show()

print("\nPER-RULE VERDICTS:")
print(pd.DataFrame(results)[["rule", "verdict", "peak_rate"]].to_string(index=False))

### ⚙️ Phase B.2: Spatial 80/20 Split (Traditional Baseline)
**Objective:** Establish the conventional Machine Learning spatial evaluation baseline.
**Logic Flow:**
* **Data Loading:** Ingest the fully populated D.2 Tabular Matrix.
* **Group Constraint:** Group all observations by their canonical `file_basename` to enforce strict spatial isolation, ensuring no single entity straddles the train/test boundary.
* **Partitioning:** Execute a `StratifiedGroupKFold` split to separate the data into an 80% Training set and a 20% Testing set.
* **Viability Check:** Verify that positive-class balance is maintained and enforce minimum refactoring event limits to guarantee the splits are mathematically viable.
* **Artifact Lock:** Export the spatial baseline sets (`b2_train_spatial`, `b2_test_spatial`) for downstream calibration.

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
import sklearn
from packaging.version import Version
from pathlib import Path

print("🚀 Starting Phase B.2: Structural Validation Split (80/20 Stratified)")

assert Version(sklearn.__version__) >= Version("1.0"), \
    f"❌ sklearn >= 1.0 required for reproducible StratifiedGroupKFold. Found: {sklearn.__version__}"

# --- Dependency Guards ---
required_vars = ["BASE_DIR", "TARGET_REPO"]
for var in required_vars:
    if var not in globals() or globals()[var] is None:
        raise RuntimeError(f"❌ Phase B.2 Dependency Error: '{var}' is missing. Did you run Phase 0?")

base_dir_p = Path(BASE_DIR)

# Auto-generate B.2 specific paths to prevent dependency crashes
train_csv_p = base_dir_p / f"b2_train_spatial_80_{TARGET_REPO}.csv"
test_csv_p = base_dir_p / f"b2_test_spatial_20_{TARGET_REPO}.csv"

# --- Fallback: Reload df_pmd if kernel restarted ---
if "df_pmd" not in globals() or df_pmd is None or df_pmd.empty:
    flat_pmd_csv_p = base_dir_p / f"pmd_flat_{TARGET_REPO}.csv"
    if flat_pmd_csv_p.exists():
        print(f"🔄 df_pmd not in memory. Reloading from {flat_pmd_csv_p.name}...")
        df_pmd = pd.read_csv(flat_pmd_csv_p)
    else:
        raise RuntimeError("❌ Error: 'df_pmd' not found in memory and flat CSV not found. Please run Phase D.2 first.")

print("\n📊 Executing Stratified Group K-Fold (Canonical Integrity at ~80/20)...")
X, y, groups = df_pmd, df_pmd["is_refactored"], df_pmd["file_basename"]

# Reverted to StratifiedGroupKFold to enforce positive-class balance across sparse data
sgkf_outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf_outer.split(X, y, groups))

df_train = df_pmd.iloc[train_idx].copy()
df_test = df_pmd.iloc[test_idx].copy()

train_pos_events = df_train[df_train["is_refactored"] == 1].groupby(["commit_sha", "file_basename"]).ngroups
test_pos_events = df_test[df_test["is_refactored"] == 1].groupby(["commit_sha", "file_basename"]).ngroups

actual_train_ratio = (len(df_train) / len(df_pmd)) * 100
print(f"   - Training Set ({actual_train_ratio:.1f}% actual): {len(df_train):,} metric rows | {train_pos_events} Unique Refactoring Events")
print(f"   - Testing Set  ({(100 - actual_train_ratio):.1f}% actual): {len(df_test):,} metric rows | {test_pos_events} Unique Refactoring Events")

# Hard kill-switches if stratification failed due to extreme sparsity
if test_pos_events == 0:
    raise RuntimeError("❌ B.2 FATAL: Test set contains zero positive refactoring events. Cannot evaluate.")
if train_pos_events == 0:
    raise RuntimeError("❌ B.2 FATAL: Training set contains zero positive events. Cannot calibrate.")

# Granular Rule Viability Report
print("\n🔍 Checking per-rule positive event densities (Rule Viability)...")
for rule in df_pmd["granular_rule"].unique():
    tr = df_train[(df_train.granular_rule == rule) & (df_train.is_refactored == 1)].shape[0]
    te = df_test[(df_test.granular_rule == rule) & (df_test.is_refactored == 1)].shape[0]
    if te < 5 or tr < 5:
        print(f"   ⚠️ {rule:<30} : train_pos={tr:<3}, test_pos={te:<3} — Likely insufficient signal for stable F1")
    else:
         print(f"   ✅ {rule:<30} : train_pos={tr:<3}, test_pos={te:<3}")

# Retaining original leakage assertion
assert len(set(df_train["file_basename"]) & set(df_test["file_basename"])) == 0, "🚨 CRITICAL LEAKAGE: A file exists in both train and test sets."

print("\n📊 Executing Inner Stratified Group Split for K-Fold IDs...")
df_train = df_train.reset_index(drop=True)
df_train["Fold_ID"] = -1

# Decorrelate inner split seed from outer split seed
sgkf_inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=99)
for fold_idx, (_, val_idx) in enumerate(sgkf_inner.split(df_train, df_train["is_refactored"], df_train["file_basename"]), 1):
    df_train.loc[val_idx, "Fold_ID"] = fold_idx

if (df_train["Fold_ID"] == -1).any():
    raise RuntimeError("❌ Phase B.2 Integrity Error: Some training rows were not assigned Fold_ID.")

assert set(df_train["Fold_ID"].unique()) == {1, 2, 3, 4, 5}, "❌ FATAL: Not all 5 folds were populated."

# Verify inner folds actually received positive examples
for fold_id in range(1, 6):
    fold_pos = df_train[(df_train["Fold_ID"] == fold_id) & (df_train["is_refactored"] == 1)].shape[0]
    if fold_pos == 0:
        print(f"   ⚠️ WARNING: Inner Fold {fold_id} has ZERO positive refactoring events. Target calibration may fail on this fold.")

print("\n💾 Saving master datasets to disk...")
df_train.to_csv(train_csv_p, index=False)
df_test.to_csv(test_csv_p, index=False)

print(f"   ✅ Saved Training Set: {train_csv_p.name}")
print(f"   ✅ Saved Testing Set:  {test_csv_p.name}")
print("\n🏁 Phase B.2 Complete. The Data Engine has secured and saved the stratified 80/20 splits!")

### ⚙️ Phase E.2: Data Scarcity Stress Test (Locked Holdout)
**Objective:** Evaluate threshold robustness under severe data volume constraints.
**Logic Flow:**
* **Baseline Alignment:** Dynamically read the 20% spatial test set generated in B.2 and lock it as the absolute, shared holdout to guarantee 1-to-1 comparability.
* **Starvation Partitioning:** Apply a secondary `StratifiedGroupKFold` directly to the baseline training data to extract a severely starved subset (retaining only ~20% of the total dataset rows).
* **Integrity Verification:** Execute a structural integrity check to ensure the starved subset still contains the absolute minimum density of positive events required for calibration.
* **Matrix Export:** Output the artificially constrained training matrix (`e2_train_scarcity`) to feed the stress-test calibration engine.

In [ ]:
# ==========================================
# PHASE E.2: DATA SCARCITY STRESS TEST (TRUE LOCKED HOLDOUT v3)
# ==========================================
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold
import sklearn
from packaging.version import Version

print("🚀 Starting Phase E.2: Data Scarcity Branch (True Locked Holdout Architecture)")

assert Version(sklearn.__version__) >= Version("1.0"), \
    f"❌ sklearn >= 1.0 required for reproducible StratifiedGroupKFold. Found: {sklearn.__version__}"

# ------------------------
# DEPENDENCY GUARD
# ------------------------
required_globals = ["TARGET_REPO", "BASE_DIR"]
for var in required_globals:
    if var not in globals() or globals()[var] is None:
        raise RuntimeError(f"❌ Phase E.2 Dependency Error: '{var}' is missing. Did you run Phase 0?")

base_dir_p = Path(BASE_DIR)
TARGET_REPO = str(TARGET_REPO)

# ------------------------
# NOTEBOOK / DATA STATE GUARD
# ------------------------
if 'df_pmd' not in globals() or df_pmd is None or df_pmd.empty:
    flat_pmd_csv_p = base_dir_p / f"pmd_flat_{TARGET_REPO}.csv"
    if flat_pmd_csv_p.exists():
        print(f"🔄 df_pmd not in memory. Reloading from {flat_pmd_csv_p.name}...")
        df_pmd = pd.read_csv(flat_pmd_csv_p)
    else:
        raise RuntimeError("❌ Error: 'df_pmd' not found in memory and flat CSV not found. Please run Phase D.2 first.")

required_cols = {"commit_sha", "file_basename", "is_refactored", "rule", "method_name", "granular_rule"}
missing_cols = required_cols - set(df_pmd.columns)
if missing_cols:
    raise RuntimeError(f"❌ Phase E.2 Error: df_pmd missing required columns: {sorted(missing_cols)}")

# Optional strictness knobs
SCARCITY_TARGET_RATIO_OF_TOTAL = 0.20   # target ~20% of total rows in train
MIN_POSITIVE_EVENTS_IN_SCARCITY = 10    # guardrail from your existing logic

# =====================================================================
# 1) Outer Split: STRICT BASENAME LOCK TO B.2'S ACTUAL SAVED TEST SET
# =====================================================================
print("\n📊 Isolating the Baseline Test Set (Extracting B.2 holdout via Basenames)...")

# Dynamically link to B.2's spatial holdout file
b2_test_path = base_dir_p / f"b2_test_spatial_20_{TARGET_REPO}.csv"
if not b2_test_path.exists():
    raise RuntimeError(f"❌ E.2 requires B.2's locked test set. Expected at: {b2_test_path.name}. Run Phase B.2 first.")

# Read B.2's test set just to get the exact set of held-out basenames
b2_test_df_raw = pd.read_csv(b2_test_path)
locked_test_basenames = set(b2_test_df_raw["file_basename"])

if not locked_test_basenames:
    raise RuntimeError("❌ E.2 Fatal: B.2's locked test set contains no basenames.")

# Reconstruct the partition directly from the live df_pmd using only the basenames.
# This prevents any row-level uniqueness bugs and guarantees data type consistency.
df_test_e2 = df_pmd[df_pmd["file_basename"].isin(locked_test_basenames)].copy().reset_index(drop=True)
df_train_full = df_pmd[~df_pmd["file_basename"].isin(locked_test_basenames)].copy().reset_index(drop=True)

if df_train_full.empty or df_test_e2.empty:
    raise RuntimeError("❌ Phase E.2 Fatal: Baseline extraction produced empty train or test set.")

# Trivial disjointness assertion to mathematically prove the lock
train_full_basenames = set(df_train_full["file_basename"])
assert len(train_full_basenames.intersection(locked_test_basenames)) == 0, "🚨 CRITICAL LEAKAGE: Locked holdout mismatch with B.2!"


# =====================================================================
# 2) Scarcity Split: Dynamically starve the training set
# =====================================================================
print("📊 Starving the Training Set (Simulating Target 20% Total Data Scarcity)...")

baseline_train_ratio_decimal = len(df_train_full) / len(df_pmd)
fraction_to_keep = SCARCITY_TARGET_RATIO_OF_TOTAL / baseline_train_ratio_decimal
fraction_to_keep = min(fraction_to_keep, 1.0)

n_splits_scarcity = max(2, int(round(1.0 / fraction_to_keep)))

n_groups_available = df_train_full['file_basename'].nunique()
assert n_groups_available >= n_splits_scarcity, \
    f"❌ FATAL: Scarcity split requires >= {n_splits_scarcity} groups, but only found {n_groups_available}."

# Outer split seed is 42 (in B.2), Scarcity slice seed is 123, Inner fold seed is 99.
sgkf_starve = StratifiedGroupKFold(n_splits=n_splits_scarcity, shuffle=True, random_state=123)

try:
    _, scarcity_idx = next(
        sgkf_starve.split(
            df_train_full,
            df_train_full["is_refactored"],
            df_train_full["file_basename"]
        )
    )
except Exception as e:
    raise RuntimeError(f"❌ Phase E.2 Split Error (Scarcity): Unable to build starvation split. Details: {e}")

df_train_e2 = df_train_full.iloc[scarcity_idx].copy().reset_index(drop=True)

# File-level subset verification
train_e2_basenames = set(df_train_e2["file_basename"])
assert train_e2_basenames.issubset(train_full_basenames), "🚨 BUG: Scarcity set contains files outside the baseline training set!"

# Event-level reporting
train_pos_events_e2 = df_train_e2[df_train_e2["is_refactored"] == 1].groupby(["commit_sha", "file_basename"]).ngroups
test_pos_events_e2 = df_test_e2[df_test_e2["is_refactored"] == 1].groupby(["commit_sha", "file_basename"]).ngroups

if test_pos_events_e2 == 0:
    raise RuntimeError("❌ E.2 FATAL: Test set contains zero positive refactoring events. Cannot evaluate.")
if train_pos_events_e2 < MIN_POSITIVE_EVENTS_IN_SCARCITY:
    raise RuntimeError(f"🚨 FATAL: Scarcity train set only has {train_pos_events_e2} positive events. Minimum required: {MIN_POSITIVE_EVENTS_IN_SCARCITY}.")

# Transparent ratio reporting
target_ratio_pct = SCARCITY_TARGET_RATIO_OF_TOTAL * 100
actual_ratio = (len(df_train_e2) / len(df_pmd)) * 100
baseline_train_ratio = baseline_train_ratio_decimal * 100
baseline_test_ratio = (len(df_test_e2) / len(df_pmd)) * 100
scarcity_within_baseline_train = (len(df_train_e2) / len(df_train_full)) * 100

train_event_density = train_pos_events_e2 / max(len(df_train_e2), 1)
test_event_density = test_pos_events_e2 / max(len(df_test_e2), 1)

print("\n🧾 E.2 SPLIT GEOMETRY EXPLAINER")
print(f"   - Baseline outer split realized: Train {baseline_train_ratio:.1f}% | Test {baseline_test_ratio:.1f}% of total rows")
print(f"   - Scarcity train is {scarcity_within_baseline_train:.1f}% of baseline-train rows")
print(f"   - Therefore scarcity train is {actual_ratio:.1f}% of total rows (target was {target_ratio_pct:.1f}%)")

print(f"\n   - Scarcity Training Set ({actual_ratio:.1f}% actual, ~{target_ratio_pct:.1f}% target): "
      f"{len(df_train_e2):,} metric rows | {train_pos_events_e2} Unique File-Release Events")
print(f"   - Locked Testing Set (constant baseline holdout): "
      f"{len(df_test_e2):,} metric rows | {test_pos_events_e2} Unique File-Release Events")
print(f"   - Scarcity event density: {train_event_density:.6f} events/row")
print(f"   - Locked-test event density: {test_event_density:.6f} events/row")


# Granular Rule Viability Report
print("\n🔍 Checking per-rule positive event densities (Rule Viability under Scarcity)...")
for rule in df_pmd["granular_rule"].unique():
    tr = df_train_e2[(df_train_e2.granular_rule == rule) & (df_train_e2.is_refactored == 1)].shape[0]
    te = df_test_e2[(df_test_e2.granular_rule == rule) & (df_test_e2.is_refactored == 1)].shape[0]
    if te < 5 or tr < 5:
        print(f"   ⚠️ {rule:<30} : train_pos={tr:<3}, test_pos={te:<3} — Likely insufficient signal for stable F1")
    else:
         print(f"   ✅ {rule:<30} : train_pos={tr:<3}, test_pos={te:<3}")


# =====================================================================
# 3) Inner Split: assign Fold_ID on scarcity training set for E.3
# =====================================================================
print("\n📊 Executing Inner Stratified Group Split for E.3 K-Fold IDs...")

df_train_e2["Fold_ID"] = -1
try:
    sgkf_inner_e2 = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=99)
    for fold_idx, (_, val_idx) in enumerate(
        sgkf_inner_e2.split(df_train_e2, df_train_e2["is_refactored"], df_train_e2["file_basename"]), 1
    ):
        df_train_e2.loc[val_idx, "Fold_ID"] = fold_idx
except Exception as e:
    raise RuntimeError(f"❌ Phase E.2 Split Error (Inner): Unable to assign 3 Fold_ID partitions on scarcity train. Details: {e}")

if (df_train_e2["Fold_ID"] == -1).any():
    raise RuntimeError("❌ Phase E.2 Integrity Error: Some scarcity-train rows were not assigned Fold_ID.")

assert set(df_train_e2["Fold_ID"].unique()) == {1, 2, 3}, "❌ FATAL: Not all 3 inner scarcity folds were populated."

for fold_id in [1, 2, 3]:
    fold_pos = df_train_e2[(df_train_e2["Fold_ID"] == fold_id) & (df_train_e2["is_refactored"] == 1)].shape[0]
    if fold_pos == 0:
        print(f"   ⚠️ WARNING: Inner scarcity Fold {fold_id} has ZERO positive events. Target calibration may fail on this fold.")

assert len(train_e2_basenames.intersection(locked_test_basenames)) == 0, "🚨 CRITICAL LEAKAGE: A file exists in both E.2 Train and Test sets!"
print("   ✅ Leakage Assertion Passed: Absolute spatial isolation confirmed.")


# =====================================================================
# 4) Export (Single Source of Truth)
# =====================================================================
E2_TRAIN_CSV_PATH = base_dir_p / f"e2_train_scarcity_{TARGET_REPO}.csv"

print("\n💾 Saving Scarcity Training dataset to disk...")
df_train_e2.to_csv(E2_TRAIN_CSV_PATH, index=False)

# We intentionally do NOT save df_test_e2 here to avoid duplicating the locked holdout.
# Downstream scripts (E.4) should directly read B.2's test CSV file.
print(f"   ✅ Saved Scarcity Training Set: {E2_TRAIN_CSV_PATH.name}")
print(f"   ✅ Locked Testing Set is preserved at: {b2_test_path.name}")
print("\n🏁 Phase E.2 Complete. The Scarcity Data Engine has secured the true locked splits!")

### ⚙️ Phase T.2: Chronological Time-Series Split
**Objective:** Implement a mathematically secure temporal evaluation free from time-travel leakage.
**Logic Flow:**
* **Timestamp Reconstruction:** Map all PMD snapshots to their exact Git commit timestamps using the lineage ledger.
* **Poisoning Prevention:** Drop any releases lacking a valid timestamp to prevent out-of-order execution.
* **Boundary Calculation:** Sort the valid releases strictly by timestamp and calculate the explicit 50% temporal midpoint.
* **Strict Temporal Partition:** Split the dataset into "The Past" (Train) and "The Future" (Test) without applying any random shuffling or spatial file grouping.
* **Mathematical Lock:** Verify the chronological boundary (`Max Past TS < Min Future TS`) and export the locked temporal matrices (`t2_train_chrono`, `t2_test_chrono`).

In [ ]:
# --- PHASE T.2: CHRONOLOGICAL TIME-SERIES SPLIT (HARDENED v3) ---
import json
import pandas as pd
from pathlib import Path
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

print("🚀 Starting Phase T.2: Chronological Branch (True Git Time-Series Split)")

# ------------------------
# DEPENDENCY GUARD
# ------------------------
required_globals = ["TARGET_REPO", "BASE_DIR", "LINEAGE_JSONL_PATH", "OFFICIAL_RELEASES_JSON_PATH"]
for var in required_globals:
    if var not in globals() or globals()[var] is None:
        raise RuntimeError(f"❌ Phase T.2 Dependency Error: '{var}' is missing or None. Did you run Phase 0?")

base_dir_p = Path(BASE_DIR)
TARGET_REPO = str(TARGET_REPO)

LINEAGE_JSONL_PATH = Path(LINEAGE_JSONL_PATH)
OFFICIAL_RELEASES_JSON_PATH = Path(OFFICIAL_RELEASES_JSON_PATH)

# Retrieve or default the split ratio matching the paper configuration (50%)
TIME_SPLIT_RATIO = globals().get("TIME_SPLIT_RATIO", 0.50)
if not (0 < TIME_SPLIT_RATIO < 1):
    raise ValueError(f"❌ Phase T.2 Error: TIME_SPLIT_RATIO must be in (0,1), got {TIME_SPLIT_RATIO}")

T2_TRAIN_CSV_PATH = base_dir_p / f"t2_train_chrono_{int(TIME_SPLIT_RATIO*100)}_{TARGET_REPO}.csv"
T2_TEST_CSV_PATH = base_dir_p / f"t2_test_chrono_{int(TIME_SPLIT_RATIO*100)}_{TARGET_REPO}.csv"
T2_TRAIN_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
T2_TEST_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

# ------------------------
# NOTEBOOK STATE GUARD & CSV FALLBACK
# ------------------------
if "df_pmd" not in globals() or df_pmd is None or df_pmd.empty:
    flat_pmd_csv_p = base_dir_p / f"pmd_flat_{TARGET_REPO}.csv"
    if flat_pmd_csv_p.exists():
        print(f"🔄 df_pmd not in memory. Reloading from {flat_pmd_csv_p.name}...")
        df_pmd = pd.read_csv(flat_pmd_csv_p)
    else:
        raise RuntimeError("❌ Phase T.2 Error: 'df_pmd' not found in memory and flat CSV not found. Run Phase D.2 first.")

if not isinstance(df_pmd, pd.DataFrame) or df_pmd.empty:
    raise RuntimeError("❌ Phase T.2 Error: df_pmd is empty or invalid even after reload.")

# Expanded required columns guard to prevent opaque downstream KeyErrors
required_cols = {"commit_sha", "is_refactored", "file_basename", "granular_rule"}
missing_cols = required_cols - set(df_pmd.columns)
if missing_cols:
    raise RuntimeError(f"❌ Phase T.2 Error: df_pmd missing required columns: {sorted(missing_cols)}")

unique_shas_t2 = df_pmd["commit_sha"].dropna().unique().tolist()
if not unique_shas_t2:
    raise RuntimeError("❌ Phase T.2 Error: No commit_sha values found in df_pmd.")

# ------------------------
# Attrition Diagnostic Check
# ------------------------
if OFFICIAL_RELEASES_JSON_PATH.exists():
    try:
        with open(OFFICIAL_RELEASES_JSON_PATH, "r", encoding="utf-8") as f:
            official_releases = json.load(f)

        if isinstance(official_releases, list):
            expected_count = len(official_releases)
        elif isinstance(official_releases, dict):
            list_lengths = [len(v) for v in official_releases.values() if isinstance(v, list)]
            expected_count = max(list_lengths) if list_lengths else len(official_releases)
        else:
            expected_count = 1

        attrition = expected_count - len(unique_shas_t2)
        print(f"\n⚠️ MSR Attrition Note: You targeted ~{expected_count} official releases from GitHub.")

        if attrition < 0:
            print(f"⚠️ PMD parsed {len(unique_shas_t2)} releases (Found {abs(attrition)} MORE than expected in JSON).\n")
        else:
            print(f"⚠️ PMD parsed {len(unique_shas_t2)} releases. The missing {attrition} failed upstream static analysis.\n")

    except Exception as e:
        print(f"\n⚠️ Could not parse rel_hist JSON for attrition check: {e}\n")
else:
    print(f"\n⚠️ Optional rel_hist file not found (skipping attrition note): {OFFICIAL_RELEASES_JSON_PATH}\n")

# ------------------------
# Rebuild commit timestamps from lineage
# ------------------------
if not LINEAGE_JSONL_PATH.exists():
    raise FileNotFoundError(f"❌ Phase T.2 Error: Missing lineage file: {LINEAGE_JSONL_PATH}")

print("⏳ Loading true Git commit timestamps to guarantee time-travel protection...")
commit_times_t2 = {}
lineage_bad_json = 0

with open(LINEAGE_JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        try:
            data = json.loads(line)
            sha = data.get("commit_sha", "")
            ts_raw = data.get("timestamp", 0)
            if sha:
                commit_times_t2[sha] = int(ts_raw or 0)
        except (json.JSONDecodeError, ValueError, TypeError):
            lineage_bad_json += 1
            continue

if lineage_bad_json:
    print(f"⚠️ Skipped malformed lineage rows: {lineage_bad_json}")

# Drop SHAs with missing/invalid timestamps
valid_shas_t2 = [sha for sha in unique_shas_t2 if commit_times_t2.get(sha, 0) > 0]
dropped_shas = len(unique_shas_t2) - len(valid_shas_t2)
if dropped_shas > 0:
    print(f"⚠️ Dropped {dropped_shas} releases missing true Git timestamps to prevent timeline poisoning.")

if len(valid_shas_t2) < 2:
    raise RuntimeError(
        f"❌ Phase T.2 Error: Need at least 2 valid timestamped releases to split; found {len(valid_shas_t2)}."
    )

# Deterministic sorting (SHA as tie-breaker for simultaneous commits)
sorted_shas_t2 = sorted(valid_shas_t2, key=lambda sha: (commit_times_t2[sha], str(sha)))
total_releases_t2 = len(sorted_shas_t2)

# Time-series split index
split_index_t2 = int(total_releases_t2 * TIME_SPLIT_RATIO)

if split_index_t2 <= 0 or split_index_t2 >= total_releases_t2:
    raise RuntimeError(
        f"❌ Phase T.2 Error: Invalid split boundary for total_releases={total_releases_t2}, "
        f"ratio={TIME_SPLIT_RATIO}. Computed split_index={split_index_t2}."
    )

train_shas_t2 = set(sorted_shas_t2[:split_index_t2])
test_shas_t2 = set(sorted_shas_t2[split_index_t2:])

boundary_sha_t2 = sorted_shas_t2[split_index_t2 - 1]
boundary_timestamp_t2 = commit_times_t2[boundary_sha_t2]

print(f"\n📊 T.2 Dataset Split Breakdown ({int(TIME_SPLIT_RATIO*100)}/{int((1-TIME_SPLIT_RATIO)*100)}):")
print(f"   - Total PMD Releases: {total_releases_t2}")
print(f"   - T.2 Training Releases (The Past):   {len(train_shas_t2)}")
print(f"   - T.2 Testing Releases  (The Future): {len(test_shas_t2)}")
print(f"   - ⏱️ TRUE Temporal Boundary Lock: Release {split_index_t2} (Git Timestamp: {boundary_timestamp_t2})\n")

df_train_t2 = df_pmd[df_pmd["commit_sha"].isin(train_shas_t2)].copy()
df_test_t2 = df_pmd[df_pmd["commit_sha"].isin(test_shas_t2)].copy()

df_train_t2["commit_time"] = df_train_t2["commit_sha"].map(commit_times_t2)
df_test_t2["commit_time"] = df_test_t2["commit_sha"].map(commit_times_t2)

if df_train_t2.empty or df_test_t2.empty:
    raise RuntimeError(
        f"❌ Phase T.2 Fatal: Empty split output. Train rows={len(df_train_t2)}, Test rows={len(df_test_t2)}."
    )

# Strict temporal leakage check with batch-release tie handling
train_max_ts = max(commit_times_t2[sha] for sha in train_shas_t2)
test_min_ts = min(commit_times_t2[sha] for sha in test_shas_t2)

if train_max_ts >= test_min_ts:
    if train_max_ts == test_min_ts:
        raise RuntimeError(
            f"❌ T.2 Boundary Tie: Last-past and first-future releases share the exact timestamp "
            f"{train_max_ts}. This is not leakage, but the boundary fell precisely on simultaneous "
            f"batch release tags. Nudge TIME_SPLIT_RATIO slightly (e.g., 0.51) to bypass the tie."
        )
    raise AssertionError("🚨 TEMPORAL LEAKAGE: Training timeline strictly overlaps future test timeline.")

# Transparent ratio reporting
actual_t2_ratio = (len(df_train_t2) / len(df_pmd)) * 100
print(f"   - T.2 Training Set ({actual_t2_ratio:.1f}% actual metric rows): {len(df_train_t2)} total rows")
print(f"   - T.2 Testing Set  ({(100 - actual_t2_ratio):.1f}% actual metric rows): {len(df_test_t2)} total rows\n")

train_refs = int(df_train_t2["is_refactored"].sum())
test_refs = int(df_test_t2["is_refactored"].sum())

print("   🔍 Temporal Imbalance Check:")
print(f"      - Past Refactored Rows:   {train_refs}")
print(f"      - Future Refactored Rows: {test_refs}\n")

# ZERO-POSITIVE KILL SWITCH (Reviewer request)
if train_refs == 0:
    raise RuntimeError("❌ T.2 FATAL: Past (training) era has zero refactored rows. Cannot calibrate a temporal threshold.")
if test_refs == 0:
    raise RuntimeError("❌ T.2 FATAL: Future (testing) era has zero refactored rows. Cannot evaluate forecasting.")

# Per-rule positive density check ported from B.2
print("🔍 Checking per-rule positive densities (Past vs Future)...")
for rule in df_pmd["granular_rule"].unique():
    tr = df_train_t2[(df_train_t2.granular_rule == rule) & (df_train_t2.is_refactored == 1)].shape[0]
    te = df_test_t2[(df_test_t2.granular_rule == rule) & (df_test_t2.is_refactored == 1)].shape[0]
    flag = "⚠️" if (tr < 5 or te < 5) else "✅"
    print(f"   {flag} {rule:<30} : past_pos={tr:<4}, future_pos={te:<4}")

# Export
df_train_t2.to_csv(T2_TRAIN_CSV_PATH, index=False)
df_test_t2.to_csv(T2_TEST_CSV_PATH, index=False)

print(f"\n💾 Saved Temporal Training Set: {T2_TRAIN_CSV_PATH.name}")
print(f"💾 Saved Temporal Testing Set: {T2_TEST_CSV_PATH.name}")
print("   ✅ Robust chronological splitting validated successfully!")
print("🏁 Phase T.2 Complete. The timeline has been permanently split (No spatial shuffling applied).")

#### 🔎 Phase T.2 Audit: Chronological Sanity & Leakage Check
**Objective:** Mathematically prove the absolute temporal isolation of the Train/Test matrices and quantify the architectural distribution shift.
**Logic Flow:**
* **Leakage Execution:** Extract the unique commit SHAs from both the Past (Train) and Future (Test) splits and execute a strict intersection test to guarantee zero mutual overlap.
* **Mathematical Time Lock:** Re-parse the Git lineage timestamps to programmatically prove the temporal boundary: `Max Past Timestamp < Min Future Timestamp`.
* **System Evolution Overlap:** Map the canonical root identities (`file_basename`) from both sets to calculate the exact percentage of the "Future" architecture that was actively present in the "Past".
* **Distribution Shift Proof:** By demonstrating that a large portion of the future system contains entirely new files unseen in the training data, this step empirically validates the necessity of chronological evaluation over naive spatial cross-validation.

In [ ]:
import json
import pandas as pd
from pathlib import Path

print("🔎 Running T.2 True Chronological Sanity Check...")

# ------------------------
# DEPENDENCY GUARD
# ------------------------
required_globals = ["TARGET_REPO", "BASE_DIR", "LINEAGE_JSONL_PATH"]
for var in required_globals:
    if var not in globals() or globals()[var] is None:
        raise RuntimeError(f"❌ Phase T.2 Audit Dependency Error: '{var}' is missing. Did you run Phase 0?")

base_dir_p = Path(BASE_DIR)
TARGET_REPO = str(TARGET_REPO)
LINEAGE_JSONL_PATH = Path(LINEAGE_JSONL_PATH)

TIME_SPLIT_RATIO = globals().get("TIME_SPLIT_RATIO", 0.50)

T2_TRAIN_CSV_PATH = base_dir_p / f"t2_train_chrono_{int(TIME_SPLIT_RATIO*100)}_{TARGET_REPO}.csv"
T2_TEST_CSV_PATH = base_dir_p / f"t2_test_chrono_{int(TIME_SPLIT_RATIO*100)}_{TARGET_REPO}.csv"

try:
    if not T2_TRAIN_CSV_PATH.exists() or not T2_TEST_CSV_PATH.exists():
        raise FileNotFoundError("Missing chronological split files. Please run Phase T.2 first.")

    check_train_t2 = pd.read_csv(T2_TRAIN_CSV_PATH)
    check_test_t2 = pd.read_csv(T2_TEST_CSV_PATH)
    print("   ✅ Split matrices successfully loaded from disk (No corruption).")

    # 1. Extract unique SHAs from both sets
    train_check_shas = set(check_train_t2["commit_sha"].dropna().unique())
    test_check_shas = set(check_test_t2["commit_sha"].dropna().unique())

    print(f"   ✅ Release counts verified (Past/Train: {len(train_check_shas)}, Future/Test: {len(test_check_shas)}).")

    # 2. The Leakage Test: Verify mutual exclusivity (Intersection should be 0)
    overlap = train_check_shas.intersection(test_check_shas)
    if len(overlap) == 0:
        print("   ✅ Zero data leakage detected (Past and Future releases are strictly mutually exclusive).")
    else:
        print(f"   ❌ LEAKAGE DETECTED: {len(overlap)} commits exist in both sets!")

    # 3. True Temporal Boundary Verification (With Kernel Restart Safe Fallback)
    if 'commit_times_t2' not in globals() or not globals()['commit_times_t2']:
        print("   🔄 'commit_times_t2' missing from memory. Re-indexing lineage timestamps on the fly...")
        commit_times_t2 = {}
        if LINEAGE_JSONL_PATH.exists():
            with open(LINEAGE_JSONL_PATH, "r", encoding="utf-8") as f:
                for line in f:
                    if not line.strip(): continue
                    try:
                        data = json.loads(line)
                        sha = data.get("commit_sha", "")
                        ts_raw = data.get("timestamp", 0)
                        if sha: commit_times_t2[sha] = int(ts_raw or 0)
                    except: continue
        globals()['commit_times_t2'] = commit_times_t2

    commit_times_t2 = globals().get('commit_times_t2', {})
    if commit_times_t2:
        train_max_time = max([commit_times_t2.get(sha, 0) for sha in train_check_shas])
        test_min_time = min([commit_times_t2.get(sha, float('inf')) for sha in test_check_shas])

        if train_max_time < test_min_time:
            print(f"   ✅ Chronological boundary mathematically verified via Git History (Max Past TS: {train_max_time} < Min Future TS: {test_min_time}).")
        else:
            print("   ❌ TEMPORAL LEAKAGE: A test release chronologically occurred before or alongside a training release!")
    else:
        print("   ⚠️ Step 3 Warning: Lineage log could not be parsed to verify structural timestamps.")

    # --- 4. OVERLAP ANALYSIS (SYSTEM EVOLUTION WITH ALIAS GRAPH) ---
    # Tracks files using their resolved Canonical Root Identity to build structural insight
    train_files = set(check_train_t2["file_basename"].dropna().unique())
    test_files = set(check_test_t2["file_basename"].dropna().unique())

    overlap_files = train_files.intersection(test_files)
    train_only = train_files - test_files
    test_only = test_files - train_files

    overlap_percent = (len(overlap_files) / len(test_files)) * 100 if len(test_files) > 0 else 0

    print("\n📁 Step 4: System Evolution & Temporal Overlap (T.2 Split)")
    print(f"      - Canonical identities in the Past (Train):            {len(train_files)}")
    print(f"      - Canonical identities in the Future (Test):           {len(test_files)}")
    print(f"      - Identities persisting across the split (Overlap):    {len(overlap_files)}")
    print(f"      - Identities retired/deleted before Future phase:      {len(train_only)}")
    print(f"      - New identities introduced in Future phase:           {len(test_only)}")
    print(f"      🎯 Percentage of Future Architecture seen in the Past: {overlap_percent:.2f}%")

except Exception as e:
    print(f"   ❌ ERROR executing chronological audit: {e}")

print("\n🏁 Audit Complete. The T.2 Chronological Split is academically secure and verified.")

## Phase 3: Threshold Calibration Engines
This phase performs multi-percentile mathematical sweeps across the training matrices to derive optimal, data-driven thresholds. It enforces rigorous statistical guardrails to ensure thresholds are actually trustworthy, rejecting metrics that perform no better than random chance.

### ⚙️ Phase B.3: Spatial K-Fold Calibration
**Objective:** Derive optimal metric thresholds using conventional spatial cross-validation.
**Logic Flow:**
* **Validation Preparation:** Partition the 80% spatial training dataset from B.2 into 5 inner validation folds.
* **Percentile Sweep:** Execute a mathematical sweep across 11 deciles (5th to 95th) of the metric value space for every target PMD rule.
* **Performance Scoring:** Calculate Precision, Recall, F1, and F0.5 scores at each percentile boundary.
* **Trust Guardrails:** Apply `MIN_FOLDS_FOR_TRUST` and `LIFT_FLOOR` (Flag-all guard) checks to statistically reject uninformative metrics.
* **Threshold Locking:** Extract the optimal `Argmax` threshold for trustworthy rules and lock them into a deterministic JSON artifact (`b3_inflection_thresholds`).

In [ ]:
import math
import pandas as pd
import warnings
import statistics
import json
from pathlib import Path
from sklearn.metrics import precision_recall_fscore_support

warnings.simplefilter(action='ignore', category=FutureWarning)

# Guard for pandas-version-specific option availability
try:
    pd.set_option('future.no_silent_downcasting', True)
except Exception:
    pass

print("🚀 Starting Phase B.3: Multi-Percentile Sweep Analysis (Optimized & Guarded)")

# ---------------------------------------------------------
# CONSTANTS & CONFIGURATION
# ---------------------------------------------------------
MIN_POS_SUPPORT = 5  # Minimum positive events required per fold to calibrate
F1_TOLERANCE = 0.01  # 1% band to resist the Recall Trap
SIGNAL_FLOOR = 0.15  # The absolute noise floor.
EPSILON = 1e-9
MIN_FOLDS_FOR_TRUST = 5  # Minimum number of folds required to consider an average stable
LIFT_FLOOR = 1.5 # Flag-all guard: lift = precision_at_threshold / base_rate. Must beat 1.5x.

# ---------------------------------------------------------
# DEPENDENCY GUARD & PATHS
# ---------------------------------------------------------
required_globals = ["TARGET_REPO", "BASE_DIR"]
for var in required_globals:
    if var not in globals() or globals()[var] is None:
        raise RuntimeError(f"❌ Phase B.3 Dependency Error: '{var}' is missing. Did you run Phase 0?")

base_dir_p = Path(BASE_DIR)
TARGET_REPO = str(TARGET_REPO)

# Dynamically bind to Phase B.2's output
TRAIN_CSV_PATH = base_dir_p / f"b2_train_spatial_80_{TARGET_REPO}.csv"
LOCKED_THRESHOLDS_PATH = base_dir_p / f"b3_inflection_thresholds_{TARGET_REPO}_locked.json"

if not TRAIN_CSV_PATH.exists():
    raise FileNotFoundError(f"❌ Phase B.3 Error: Could not find {TRAIN_CSV_PATH.name}. You must run Phase B.2 first!")

df_train = pd.read_csv(TRAIN_CSV_PATH)

# Required-column guardrails
required_cols = {
    "commit_sha", "file_basename", "method_name",
    "is_refactored", "granular_rule", "metric_value", "Fold_ID"
}
missing_cols = required_cols - set(df_train.columns)
if missing_cols:
    raise RuntimeError(f"❌ Phase B.3 Error: Missing required columns: {sorted(missing_cols)}")

train_basenames_set = set(df_train["file_basename"].dropna().unique())
print(f"📊 Loaded Training Set with {len(train_basenames_set)} canonical entities ready for K-Fold Calibration.")

# ---------------------------------------------------------
# 2. METHOD COUNT FOR TooManyMethods
# ---------------------------------------------------------
if "Calculated_MethodCount" not in df_train.columns:
    raise RuntimeError("❌ Phase B.3: 'Calculated_MethodCount' missing — re-run D.2 (it now computes this) then B.2.")

# ---------------------------------------------------------
# THE K-FOLD CALIBRATION ENGINE
# ---------------------------------------------------------
fold_ids = sorted(df_train["Fold_ID"].dropna().unique())
if not fold_ids:
    raise RuntimeError("❌ Phase B.3 Error: No Fold_ID values found. Run Phase B.2 to assign folds first.")
print(f"\n📈 Executing {len(fold_ids)}-Fold Cross-Validation Tournament...")

sweep_percentiles = [0.05, 0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85, 0.90, 0.95]

rules_to_calc = [
    "NcssCount_Class", "CyclomaticComplexity_Class", "ExcessivePublicCount",
    "CouplingBetweenObjects", "ExcessiveImports", "NcssCount_Method",
    "CyclomaticComplexity_Method", "TooManyMethods"
]

all_rules_in_data = set(df_train["granular_rule"].dropna().unique())
uncalibrated = all_rules_in_data - set(rules_to_calc)
if uncalibrated:
    print(f"⚠️ Note: {len(uncalibrated)} raw PMD rules skipped (intentionally omitted from calibration): {uncalibrated}")

rule_performance = {rule: {ptile: [] for ptile in sweep_percentiles} for rule in rules_to_calc}

for val_fold in fold_ids:
    print(f"   ⚙️ Processing Fold {val_fold} as Validation...")

    train_k = df_train[df_train["Fold_ID"] != val_fold].copy()
    val_k = df_train[df_train["Fold_ID"] == val_fold].copy()

    for rule in rules_to_calc:
        if rule == "TooManyMethods":
            GOD_CLASS_RULES = ["NcssCount_Class", "CyclomaticComplexity_Class",
                               "ExcessivePublicCount", "TooManyMethods"]
            k_train_data = (train_k[train_k["granular_rule"].isin(GOD_CLASS_RULES)]
                            .drop_duplicates(subset=["commit_sha", "file_basename"]))
            k_val_data = (val_k[val_k["granular_rule"].isin(GOD_CLASS_RULES)]
                          .drop_duplicates(subset=["commit_sha", "file_basename"]).copy())
            k_val_data["val"] = k_val_data["Calculated_MethodCount"]
            metric_col = "Calculated_MethodCount"
        else:
            k_train_data = train_k[train_k["granular_rule"] == rule]
            k_val_data = val_k[val_k["granular_rule"] == rule].copy()
            k_val_data["val"] = k_val_data["metric_value"]
            metric_col = "metric_value"

        if k_val_data.empty:
            continue

        # Minimum Support Guard for Training
        bp = k_train_data[k_train_data["is_refactored"] == 1][metric_col]
        if len(bp) < MIN_POS_SUPPORT:
            continue

        # Pin linear interpolation for perfect reproducibility
        quantiles = bp.quantile(sweep_percentiles, interpolation="linear").to_dict()

        # Minimum Support Guard for Validation
        v_actual_int = k_val_data["is_refactored"].astype(int)
        if v_actual_int.sum() < MIN_POS_SUPPORT:
            continue

        for ptile, thresh in quantiles.items():
            ithresh = int(math.ceil(thresh))
            v_pred = (k_val_data["val"] >= ithresh).astype(int)

            p, r, f1, _ = precision_recall_fscore_support(
                v_actual_int, v_pred, average="binary", zero_division=0
            )

            f0_5 = ((1 + 0.5**2) * (p * r)) / ((0.5**2 * p) + r + 1e-9)

            rule_performance[rule][ptile].append({
                "thresh": ithresh, "p": p, "r": r, "f1": f1, "f0_5": f0_5,
                "base": float(v_actual_int.mean())  # base rate = precision of flag-all
            })

# ---------------------------------------------------------
# PRINT THE SWEEP MATRICES
# ---------------------------------------------------------
matrix_rows_f1 = []
matrix_rows_f0_5 = []

for rule in rules_to_calc:
    row_f1 = {"Rule": rule}
    row_f0_5 = {"Rule": rule}
    for ptile in sweep_percentiles:
        results = rule_performance[rule][ptile]
        if results:
            avg_p = sum(res["p"] for res in results) / len(results)
            avg_r = sum(res["r"] for res in results) / len(results)
            avg_f1 = sum(res["f1"] for res in results) / len(results)
            avg_f0_5 = sum(res["f0_5"] for res in results) / len(results)
            med_thresh = int(math.ceil(statistics.median(res["thresh"] for res in results)))

            row_f1[f"{int(ptile*100)}th"] = f"{med_thresh} (P:{avg_p:.2f}|R:{avg_r:.2f}|F1:{avg_f1:.2f})"
            row_f0_5[f"{int(ptile*100)}th"] = f"{med_thresh} (P:{avg_p:.2f}|R:{avg_r:.2f}|F0.5:{avg_f0_5:.2f})"
        else:
            row_f1[f"{int(ptile*100)}th"] = "N/A"
            row_f0_5[f"{int(ptile*100)}th"] = "N/A"

    matrix_rows_f1.append(row_f1)
    matrix_rows_f0_5.append(row_f0_5)

print("\n🔍 Standard F1 Sweep Matrix [ Int Threshold (P | R | F1) ]:")
print(pd.DataFrame(matrix_rows_f1).to_markdown(index=False))

print("\n🎯 Precision-Optimized Sweep Matrix [ Int Threshold (P | R | F0.5) ]:")
print(pd.DataFrame(matrix_rows_f0_5).to_markdown(index=False))

# ---------------------------------------------------------
# AGGREGATE & LOCK DUAL THRESHOLDS (STRICT ARGMAX + STABILITY + TRUST)
# ---------------------------------------------------------
locked_f1_thresholds = {}
locked_f0_5_thresholds = {}

for rule in rules_to_calc:
    # 1. Track Strict Argmax (The New Default)
    argmax_avg_f1, argmax_ptile_f1, argmax_thresh_f1 = -1.0, None, None
    argmax_avg_f0_5, argmax_ptile_f0_5, argmax_thresh_f0_5 = -1.0, None, None
    # Precision & base rate at the winning percentile (for the flag-all lift guard)
    argmax_p_f1, argmax_base_f1 = 0.0, 0.0
    argmax_p_f0_5, argmax_base_f0_5 = 0.0, 0.0

    # 2. Track Tolerance Band (For Stability Comparison)
    band_avg_f1, band_ptile_f1, band_thresh_f1 = -1.0, None, None
    band_avg_f0_5, band_ptile_f0_5, band_thresh_f0_5 = -1.0, None, None

    winning_results_count_f1 = 0
    winning_results_count_f0_5 = 0

    for ptile in sweep_percentiles:
        results = rule_performance[rule][ptile]
        if not results:
            continue

        avg_f1 = sum(res["f1"] for res in results) / len(results)
        avg_f0_5 = sum(res["f0_5"] for res in results) / len(results)
        avg_p = sum(res["p"] for res in results) / len(results)
        avg_base = sum(res.get("base", 0.0) for res in results) / len(results)
        median_thresh = int(math.ceil(statistics.median(res["thresh"] for res in results)))

        # ----- Strict Argmax Logic (New Default) -----
        if avg_f1 > argmax_avg_f1 + EPSILON:
            argmax_avg_f1 = avg_f1
            argmax_ptile_f1 = f"{int(ptile*100)}th"
            argmax_thresh_f1 = median_thresh
            winning_results_count_f1 = len(results)
            argmax_p_f1 = avg_p
            argmax_base_f1 = avg_base

        if avg_f0_5 > argmax_avg_f0_5 + EPSILON:
            argmax_avg_f0_5 = avg_f0_5
            argmax_ptile_f0_5 = f"{int(ptile*100)}th"
            argmax_thresh_f0_5 = median_thresh
            winning_results_count_f0_5 = len(results)
            argmax_p_f0_5 = avg_p
            argmax_base_f0_5 = avg_base

        # ----- Tolerance Band Logic (Used strictly as a Noise Detector) -----
        within_band_f1 = avg_f1 >= (band_avg_f1 - F1_TOLERANCE)
        prev_band_f1 = band_thresh_f1 if band_thresh_f1 is not None else float("-inf")

        if (avg_f1 > band_avg_f1 + EPSILON) or (within_band_f1 and median_thresh > prev_band_f1):
            band_avg_f1 = avg_f1
            band_ptile_f1 = f"{int(ptile*100)}th"
            band_thresh_f1 = median_thresh

        within_band_f0_5 = avg_f0_5 >= (band_avg_f0_5 - F1_TOLERANCE)
        prev_band_f0_5 = band_thresh_f0_5 if band_thresh_f0_5 is not None else float("-inf")

        if (avg_f0_5 > band_avg_f0_5 + EPSILON) or (within_band_f0_5 and median_thresh > prev_band_f0_5):
            band_avg_f0_5 = avg_f0_5
            band_ptile_f0_5 = f"{int(ptile*100)}th"
            band_thresh_f0_5 = median_thresh

    # ----- 3. Determine Stability & Trustworthiness -----
    is_stable_f1 = False
    if argmax_thresh_f1 is not None and band_thresh_f1 is not None:
        max_t = max(argmax_thresh_f1, band_thresh_f1)
        min_t = max(min(argmax_thresh_f1, band_thresh_f1), 1) # Prevent div by zero
        is_stable_f1 = (max_t / min_t) <= 2.0

    # F1 Trust Logic
    signal_ok_f1 = (argmax_avg_f1 != -1.0) and (argmax_avg_f1 >= SIGNAL_FLOOR)
    enough_folds_f1 = winning_results_count_f1 >= MIN_FOLDS_FOR_TRUST
    precision_lift_f1 = (argmax_p_f1 / argmax_base_f1) if argmax_base_f1 > EPSILON else 0.0
    lift_ok_f1 = precision_lift_f1 >= LIFT_FLOOR  # rejects flag-all (lift ≈ 1.0)
    trustworthy_f1 = is_stable_f1 and signal_ok_f1 and enough_folds_f1 and lift_ok_f1

    is_stable_f0_5 = False
    if argmax_thresh_f0_5 is not None and band_thresh_f0_5 is not None:
        max_t = max(argmax_thresh_f0_5, band_thresh_f0_5)
        min_t = max(min(argmax_thresh_f0_5, band_thresh_f0_5), 1)
        is_stable_f0_5 = (max_t / min_t) <= 2.0

    # F0.5 Trust Logic
    signal_ok_f0_5 = (argmax_avg_f0_5 != -1.0) and (argmax_avg_f0_5 >= SIGNAL_FLOOR)
    enough_folds_f0_5 = winning_results_count_f0_5 >= MIN_FOLDS_FOR_TRUST
    precision_lift_f0_5 = (argmax_p_f0_5 / argmax_base_f0_5) if argmax_base_f0_5 > EPSILON else 0.0
    lift_ok_f0_5 = precision_lift_f0_5 >= LIFT_FLOOR
    trustworthy_f0_5 = is_stable_f0_5 and signal_ok_f0_5 and enough_folds_f0_5 and lift_ok_f0_5

    locked_f1_thresholds[rule] = {
        "calibrated": argmax_thresh_f1 is not None,
        "threshold_stable": is_stable_f1,
        "trustworthy": trustworthy_f1,
        "percentile": argmax_ptile_f1 if argmax_ptile_f1 else "Default",
        "threshold_value": int(argmax_thresh_f1) if argmax_thresh_f1 is not None else None,
        "band_divergence": int(band_thresh_f1) if band_thresh_f1 is not None else None,
        "kfold_avg_f1": round(argmax_avg_f1, 4) if argmax_avg_f1 != -1.0 else None,
        "precision_lift": round(precision_lift_f1, 2),
        "base_rate": round(argmax_base_f1, 4),
        "folds_used": winning_results_count_f1
    }

    locked_f0_5_thresholds[rule] = {
        "calibrated": argmax_thresh_f0_5 is not None,
        "threshold_stable": is_stable_f0_5,
        "trustworthy": trustworthy_f0_5,
        "percentile": argmax_ptile_f0_5 if argmax_ptile_f0_5 else "Default",
        "threshold_value": int(argmax_thresh_f0_5) if argmax_thresh_f0_5 is not None else None,
        "band_divergence": int(band_thresh_f0_5) if band_thresh_f0_5 is not None else None,
        "kfold_avg_f0_5": round(argmax_avg_f0_5, 4) if argmax_avg_f0_5 != -1.0 else None,
        "precision_lift": round(precision_lift_f0_5, 2),
        "base_rate": round(argmax_base_f0_5, 4),
        "folds_used": winning_results_count_f0_5
    }

# ---------------------------------------------------------
# PRINT LOCKED THRESHOLDS & SAVE TO JSON
# ---------------------------------------------------------
print("\n🏆 F1-Optimized Thresholds:")
print(pd.DataFrame(locked_f1_thresholds).T.to_markdown())

print("\n🎯 Precision-Optimized (F0.5) Thresholds:")
print(pd.DataFrame(locked_f0_5_thresholds).T.to_markdown())

# Backward-Compatible JSON Schema
final_json_export = locked_f1_thresholds.copy()
final_json_export["_precision_optimized_f0_5"] = locked_f0_5_thresholds

# Self-describing metadata block
final_json_export["_meta"] = {
    "chosen_n_splits": len(fold_ids),
    "min_pos_support": MIN_POS_SUPPORT,
    "min_folds_for_trust": MIN_FOLDS_FOR_TRUST,
    "lift_floor": LIFT_FLOOR,
    "f1_tolerance_band": F1_TOLERANCE,
    "quantile_interpolation": "linear"
}

with open(LOCKED_THRESHOLDS_PATH, "w", encoding="utf-8") as f:
    json.dump(final_json_export, f, indent=4)

print(f"\n💾 Saved Dual Baseline locked thresholds to disk: {LOCKED_THRESHOLDS_PATH}")
print("🏁 Phase B.3 Complete.")

### ⚙️ Phase E.3: Scarcity K-Fold Calibration
**Objective:** Derive thresholds under extreme data starvation.
**Logic Flow:**
* **Constrained Ingestion:** Consume the ~20% artificially starved training matrix from Phase E.2.
* **Fold Adjustment:** Adjust inner cross-validation constraints (reducing `MIN_FOLDS_FOR_TRUST` to 3) to accommodate the heavily reduced spatial volume.
* **Sweeping & Scoring:** Execute the exact same multi-percentile sweep and F-score calibration logic as utilized in Phase B.3.
* **Resilience Testing:** Determine if the statistical guardrails reject the metric entirely due to noise, or if a stable threshold survives the scarcity.
* **Threshold Locking:** Save the scarcity-constrained outputs to the `e3_inflection_thresholds` JSON artifact.

In [ ]:
import math
import pandas as pd
import warnings
import statistics
import json
from pathlib import Path
from sklearn.metrics import precision_recall_fscore_support

warnings.simplefilter(action='ignore', category=FutureWarning)

# Guard for pandas-version-specific option availability
try:
    pd.set_option('future.no_silent_downcasting', True)
except Exception:
    pass

print("🚀 Starting Phase E.3: Multi-Percentile Sweep Analysis (Scarcity Edition)")

# ---------------------------------------------------------
# CONSTANTS & CONFIGURATION (PORTED FROM B.3)
# ---------------------------------------------------------
MIN_POS_SUPPORT = 5  # Minimum positive events required per fold to calibrate
F1_TOLERANCE = 0.01  # 1% band to resist the Recall Trap
# The absolute noise floor. An F-score below this is statistically indistinguishable from random guessing.
SIGNAL_FLOOR = 0.15
EPSILON = 1e-9
MIN_FOLDS_FOR_TRUST = 3  # Minimum number of folds required to consider an average stable
# Flag-all guard: the certified threshold's precision must beat the base rate
# (= precision of the trivial "flag everything" classifier) by this factor.
# precision_lift = precision_at_threshold / base_rate. Flag-all => lift ≈ 1.0 => rejected.
LIFT_FLOOR = 1.5

# 1. Path Resolution & Data Loading
BASE_DIR_PATH = Path(globals().get("BASE_DIR", "."))
TARGET_REPO = globals().get("TARGET_REPO", "unknown_repo")

# Respect the global variable from E.2 if it exists, fallback to default if not
E2_TRAIN_CSV_PATH = Path(
    globals().get("E2_TRAIN_CSV_PATH", BASE_DIR_PATH / f"e2_train_scarcity_{TARGET_REPO}.csv")
)
if not E2_TRAIN_CSV_PATH.exists():
    raise FileNotFoundError(
        f"❌ Phase E.3 Error: Could not find {E2_TRAIN_CSV_PATH.name}. You must run Phase E.2 first!"
    )

SCARCITY_THRESHOLDS_PATH = BASE_DIR_PATH / f"e3_inflection_thresholds_{TARGET_REPO}_scarcity.json"
SCARCITY_THRESHOLDS_PATH.parent.mkdir(parents=True, exist_ok=True)

df_train_e2 = pd.read_csv(E2_TRAIN_CSV_PATH)

# Required-column guardrails
required_cols = {
    "commit_sha", "file_basename", "method_name",
    "is_refactored", "granular_rule", "metric_value", "Fold_ID"
}
missing_cols = required_cols - set(df_train_e2.columns)
if missing_cols:
    raise RuntimeError(f"❌ Phase E.3 Error: Missing required columns: {sorted(missing_cols)}")

train_basenames_set = set(df_train_e2["file_basename"].dropna().unique())
print(f"📊 Loaded Scarcity Training Set (20%) with {len(train_basenames_set)} canonical entities.")

# ---------------------------------------------------------
# 2. METHOD COUNT FOR TooManyMethods — now provided by D.2 and carried through
#    the split CSVs, so every phase reads an identical value. Just guard it.
# ---------------------------------------------------------
if "Calculated_MethodCount" not in df_train_e2.columns:
    raise RuntimeError("❌ Phase E.3: 'Calculated_MethodCount' missing — re-run D.2 (it now computes this) then E.2.")


# ---------------------------------------------------------
# THE K-FOLD CALIBRATION ENGINE
# ---------------------------------------------------------
fold_ids = sorted(df_train_e2["Fold_ID"].dropna().unique())
if not fold_ids:
    raise RuntimeError("❌ Phase E.3 Error: No Fold_ID values found. Run Phase E.2 to assign folds first.")
print(f"\n📈 Executing {len(fold_ids)}-Fold Cross-Validation Tournament (Scarcity)...")

sweep_percentiles = [0.05, 0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85, 0.90, 0.95]

rules_to_calc = [
    "NcssCount_Class", "CyclomaticComplexity_Class", "ExcessivePublicCount",
    "CouplingBetweenObjects", "ExcessiveImports", "NcssCount_Method",
    "CyclomaticComplexity_Method", "TooManyMethods"
]

all_rules_in_data = set(df_train_e2["granular_rule"].dropna().unique())
uncalibrated = all_rules_in_data - set(rules_to_calc)
if uncalibrated:
    print(f"⚠️ Note: {len(uncalibrated)} raw PMD rules skipped (intentionally omitted from calibration): {uncalibrated}")

rule_performance = {rule: {ptile: [] for ptile in sweep_percentiles} for rule in rules_to_calc}

for val_fold in fold_ids:
    print(f"   ⚙️ Processing Fold {val_fold} as Validation...")

    train_k = df_train_e2[df_train_e2["Fold_ID"] != val_fold].copy()
    val_k = df_train_e2[df_train_e2["Fold_ID"] == val_fold].copy()

    for rule in rules_to_calc:
        if rule == "TooManyMethods":
            # Per-CLASS metric: ONE row per class, God-Class label only — NOT every
            # class_level row (which duplicates each class and mixes in the
            # Feature-Envy-labelled CBO/Imports rows).
            GOD_CLASS_RULES = ["NcssCount_Class", "CyclomaticComplexity_Class",
                               "ExcessivePublicCount", "TooManyMethods"]
            k_train_data = (train_k[train_k["granular_rule"].isin(GOD_CLASS_RULES)]
                            .drop_duplicates(subset=["commit_sha", "file_basename"]))
            k_val_data = (val_k[val_k["granular_rule"].isin(GOD_CLASS_RULES)]
                          .drop_duplicates(subset=["commit_sha", "file_basename"]).copy())
            k_val_data["val"] = k_val_data["Calculated_MethodCount"]
            metric_col = "Calculated_MethodCount"
        else:
            k_train_data = train_k[train_k["granular_rule"] == rule]
            k_val_data = val_k[val_k["granular_rule"] == rule].copy()
            k_val_data["val"] = k_val_data["metric_value"]
            metric_col = "metric_value"

        if k_val_data.empty:
            continue

        # FIX 3: Minimum Support Guard for Training (Ported from B.3)
        bp = k_train_data[k_train_data["is_refactored"] == 1][metric_col]
        if len(bp) < MIN_POS_SUPPORT:
            # print(f"  ⚠️ Fold {val_fold} | {rule}: Only {len(bp)} positives in scarcity train. Skipping.")
            continue

        # MINOR FIX: Pin linear interpolation for perfect reproducibility
        quantiles = bp.quantile(sweep_percentiles, interpolation="linear").to_dict()

        # FIX 3: Minimum Support Guard for Validation (Ported from B.3)
        v_actual_int = k_val_data["is_refactored"].astype(int)
        if v_actual_int.sum() < MIN_POS_SUPPORT:
            # print(f"  ⚠️ Fold {val_fold} | {rule}: Only {v_actual_int.sum()} positives in scarcity val. Skipping.")
            continue

        for ptile, thresh in quantiles.items():
            # enforce integer threshold at scoring time
            ithresh = int(math.ceil(thresh))
            v_pred = (k_val_data["val"] >= ithresh).astype(int)

            p, r, f1, _ = precision_recall_fscore_support(
                v_actual_int, v_pred, average="binary", zero_division=0
            )

            # Explicit F0.5 calculation with zero-division guard
            f0_5 = ((1 + 0.5**2) * (p * r)) / ((0.5**2 * p) + r + 1e-9)

            rule_performance[rule][ptile].append({
                "thresh": ithresh, "p": p, "r": r, "f1": f1, "f0_5": f0_5,
                "base": float(v_actual_int.mean())  # base rate = precision of flag-all
            })

# ---------------------------------------------------------
# PRINT THE SWEEP MATRICES
# ---------------------------------------------------------
matrix_rows_f1 = []
matrix_rows_f0_5 = []

for rule in rules_to_calc:
    row_f1 = {"Rule": rule}
    row_f0_5 = {"Rule": rule}
    for ptile in sweep_percentiles:
        results = rule_performance[rule][ptile]
        if results:
            avg_p = sum(res["p"] for res in results) / len(results)
            avg_r = sum(res["r"] for res in results) / len(results)
            avg_f1 = sum(res["f1"] for res in results) / len(results)
            avg_f0_5 = sum(res["f0_5"] for res in results) / len(results)
            med_thresh = int(math.ceil(statistics.median(res["thresh"] for res in results)))

            row_f1[f"{int(ptile*100)}th"] = f"{med_thresh} (P:{avg_p:.2f}|R:{avg_r:.2f}|F1:{avg_f1:.2f})"
            row_f0_5[f"{int(ptile*100)}th"] = f"{med_thresh} (P:{avg_p:.2f}|R:{avg_r:.2f}|F0.5:{avg_f0_5:.2f})"
        else:
            row_f1[f"{int(ptile*100)}th"] = "N/A"
            row_f0_5[f"{int(ptile*100)}th"] = "N/A"

    matrix_rows_f1.append(row_f1)
    matrix_rows_f0_5.append(row_f0_5)

print("\n🔍 Standard F1 Sweep Matrix [ Int Threshold (P | R | F1) ]:")
print(pd.DataFrame(matrix_rows_f1).to_markdown(index=False))

print("\n🎯 Precision-Optimized Sweep Matrix [ Int Threshold (P | R | F0.5) ]:")
print(pd.DataFrame(matrix_rows_f0_5).to_markdown(index=False))

# ---------------------------------------------------------
# AGGREGATE & LOCK DUAL THRESHOLDS (STRICT ARGMAX + STABILITY + TRUST)
# ---------------------------------------------------------
locked_f1_thresholds = {}
locked_f0_5_thresholds = {}

for rule in rules_to_calc:
    # 1. Track Strict Argmax (The New Default)
    argmax_avg_f1, argmax_ptile_f1, argmax_thresh_f1 = -1.0, None, None
    argmax_avg_f0_5, argmax_ptile_f0_5, argmax_thresh_f0_5 = -1.0, None, None
    # Precision & base rate at the winning percentile (for the flag-all lift guard)
    argmax_p_f1, argmax_base_f1 = 0.0, 0.0
    argmax_p_f0_5, argmax_base_f0_5 = 0.0, 0.0

    # 2. Track Tolerance Band (For Stability Comparison)
    band_avg_f1, band_ptile_f1, band_thresh_f1 = -1.0, None, None
    band_avg_f0_5, band_ptile_f0_5, band_thresh_f0_5 = -1.0, None, None

    winning_results_count_f1 = 0
    winning_results_count_f0_5 = 0

    for ptile in sweep_percentiles:
        results = rule_performance[rule][ptile]
        if not results:
            continue

        avg_f1 = sum(res["f1"] for res in results) / len(results)
        avg_f0_5 = sum(res["f0_5"] for res in results) / len(results)
        avg_p = sum(res["p"] for res in results) / len(results)
        avg_base = sum(res.get("base", 0.0) for res in results) / len(results)
        median_thresh = int(math.ceil(statistics.median(res["thresh"] for res in results)))

        # ----- Strict Argmax Logic (New Default) -----
        if avg_f1 > argmax_avg_f1 + EPSILON:
            argmax_avg_f1 = avg_f1
            argmax_ptile_f1 = f"{int(ptile*100)}th"
            argmax_thresh_f1 = median_thresh
            winning_results_count_f1 = len(results)
            argmax_p_f1 = avg_p
            argmax_base_f1 = avg_base

        if avg_f0_5 > argmax_avg_f0_5 + EPSILON:
            argmax_avg_f0_5 = avg_f0_5
            argmax_ptile_f0_5 = f"{int(ptile*100)}th"
            argmax_thresh_f0_5 = median_thresh
            winning_results_count_f0_5 = len(results)
            argmax_p_f0_5 = avg_p
            argmax_base_f0_5 = avg_base

        # ----- Tolerance Band Logic (Used strictly as a Noise Detector) -----
        within_band_f1 = avg_f1 >= (band_avg_f1 - F1_TOLERANCE)
        prev_band_f1 = band_thresh_f1 if band_thresh_f1 is not None else float("-inf")

        if (avg_f1 > band_avg_f1 + EPSILON) or (within_band_f1 and median_thresh > prev_band_f1):
            band_avg_f1 = avg_f1
            band_ptile_f1 = f"{int(ptile*100)}th"
            band_thresh_f1 = median_thresh

        within_band_f0_5 = avg_f0_5 >= (band_avg_f0_5 - F1_TOLERANCE)
        prev_band_f0_5 = band_thresh_f0_5 if band_thresh_f0_5 is not None else float("-inf")

        if (avg_f0_5 > band_avg_f0_5 + EPSILON) or (within_band_f0_5 and median_thresh > prev_band_f0_5):
            band_avg_f0_5 = avg_f0_5
            band_ptile_f0_5 = f"{int(ptile*100)}th"
            band_thresh_f0_5 = median_thresh

    # ----- 3. Determine Stability & Trustworthiness -----
    is_stable_f1 = False
    if argmax_thresh_f1 is not None and band_thresh_f1 is not None:
        max_t = max(argmax_thresh_f1, band_thresh_f1)
        min_t = max(min(argmax_thresh_f1, band_thresh_f1), 1) # Prevent div by zero
        is_stable_f1 = (max_t / min_t) <= 2.0

    # F1 Trust Logic
    signal_ok_f1 = (argmax_avg_f1 != -1.0) and (argmax_avg_f1 >= SIGNAL_FLOOR)
    enough_folds_f1 = winning_results_count_f1 >= MIN_FOLDS_FOR_TRUST
    precision_lift_f1 = (argmax_p_f1 / argmax_base_f1) if argmax_base_f1 > EPSILON else 0.0
    lift_ok_f1 = precision_lift_f1 >= LIFT_FLOOR  # rejects flag-all (lift ≈ 1.0)
    trustworthy_f1 = is_stable_f1 and signal_ok_f1 and enough_folds_f1 and lift_ok_f1

    is_stable_f0_5 = False
    if argmax_thresh_f0_5 is not None and band_thresh_f0_5 is not None:
        max_t = max(argmax_thresh_f0_5, band_thresh_f0_5)
        min_t = max(min(argmax_thresh_f0_5, band_thresh_f0_5), 1)
        is_stable_f0_5 = (max_t / min_t) <= 2.0

    # F0.5 Trust Logic
    signal_ok_f0_5 = (argmax_avg_f0_5 != -1.0) and (argmax_avg_f0_5 >= SIGNAL_FLOOR)
    enough_folds_f0_5 = winning_results_count_f0_5 >= MIN_FOLDS_FOR_TRUST
    precision_lift_f0_5 = (argmax_p_f0_5 / argmax_base_f0_5) if argmax_base_f0_5 > EPSILON else 0.0
    lift_ok_f0_5 = precision_lift_f0_5 >= LIFT_FLOOR
    trustworthy_f0_5 = is_stable_f0_5 and signal_ok_f0_5 and enough_folds_f0_5 and lift_ok_f0_5

    locked_f1_thresholds[rule] = {
        "calibrated": argmax_thresh_f1 is not None,
        "threshold_stable": is_stable_f1,
        "trustworthy": trustworthy_f1,
        "percentile": argmax_ptile_f1 if argmax_ptile_f1 else "Default",
        "threshold_value": int(argmax_thresh_f1) if argmax_thresh_f1 is not None else None,
        "band_divergence": int(band_thresh_f1) if band_thresh_f1 is not None else None,
        "kfold_avg_f1": round(argmax_avg_f1, 4) if argmax_avg_f1 != -1.0 else None,
        "precision_lift": round(precision_lift_f1, 2),
        "base_rate": round(argmax_base_f1, 4),
        "folds_used": winning_results_count_f1
    }

    locked_f0_5_thresholds[rule] = {
        "calibrated": argmax_thresh_f0_5 is not None,
        "threshold_stable": is_stable_f0_5,
        "trustworthy": trustworthy_f0_5,
        "percentile": argmax_ptile_f0_5 if argmax_ptile_f0_5 else "Default",
        "threshold_value": int(argmax_thresh_f0_5) if argmax_thresh_f0_5 is not None else None,
        "band_divergence": int(band_thresh_f0_5) if band_thresh_f0_5 is not None else None,
        "kfold_avg_f0_5": round(argmax_avg_f0_5, 4) if argmax_avg_f0_5 != -1.0 else None,
        "precision_lift": round(precision_lift_f0_5, 2),
        "base_rate": round(argmax_base_f0_5, 4),
        "folds_used": winning_results_count_f0_5
    }

# ---------------------------------------------------------
# PRINT LOCKED THRESHOLDS & SAVE TO JSON
# ---------------------------------------------------------
print("\n🏆 F1-Optimized Scarcity Thresholds:")
print(pd.DataFrame(locked_f1_thresholds).T.to_markdown())

print("\n🎯 Precision-Optimized (F0.5) Scarcity Thresholds:")
print(pd.DataFrame(locked_f0_5_thresholds).T.to_markdown())

# Backward-Compatible JSON Schema
final_json_export = locked_f1_thresholds.copy()
final_json_export["_precision_optimized_f0_5"] = locked_f0_5_thresholds

# MINOR FIX: Self-describing metadata block
final_json_export["_meta"] = {
    "chosen_n_splits": len(fold_ids),
    "min_pos_support": MIN_POS_SUPPORT,
    "min_folds_for_trust": MIN_FOLDS_FOR_TRUST,
    "lift_floor": LIFT_FLOOR,
    "f1_tolerance_band": F1_TOLERANCE,
    "quantile_interpolation": "linear"
}

with open(SCARCITY_THRESHOLDS_PATH, "w", encoding="utf-8") as f:
    json.dump(final_json_export, f, indent=4)

print(f"\n💾 Saved Dual Baseline locked thresholds to disk: {SCARCITY_THRESHOLDS_PATH}")
print("🏁 Phase E.3 Complete.")

### ⚙️ Phase T.3: Chronological K-Fold Calibration
**Objective:** Derive thresholds using a secure rolling time-window to simulate real-world deployment.
**Logic Flow:**
* **Historical Ingestion:** Load the "Past" timeline dataset from Phase T.2.
* **Time-Series Generation:** Apply a strict `TimeSeriesSplit` to generate expanding chronological validation folds, guaranteeing that the model only uses historical data to predict its immediate future.
* **Temporal Sweeping:** Run the percentile calibration engine across these strict time boundaries.
* **Sparsity Guarding:** Enforce rigorous positive-support checks to prevent early historical folds from hallucinating thresholds on overly sparse timelines.
* **Threshold Locking:** Lock the temporally derived boundaries into the `t3_inflection_thresholds` JSON artifact.

In [ ]:
import math
import pandas as pd
import warnings
import statistics
import json
import sklearn
import numpy as np
from pathlib import Path
from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import TimeSeriesSplit
from packaging.version import Version

warnings.simplefilter(action='ignore', category=FutureWarning)

# Sklearn version guard for safe temporal splitting
assert Version(sklearn.__version__) >= Version("1.0"), \
    f"❌ sklearn >= 1.0 required for TimeSeriesSplit. Found: {sklearn.__version__}"

# Guard for pandas-version-specific option availability
try:
    pd.set_option('future.no_silent_downcasting', True)
except Exception:
    pass

print("🚀 Starting Phase T.3: Chronological K-Fold Calibration Engine")

# ---------------------------------------------------------
# CONSTANTS & CONFIGURATION (PORTED FROM B.3 / E.3)
# ---------------------------------------------------------
MIN_POS_SUPPORT_VAL = 5    # Require 5 positives to score validation
MIN_POS_SUPPORT_TRAIN = 2  # Require 2 positives (minimum needed for percentiles) to allow early temporal windows
F1_TOLERANCE = 0.01        # 1% band to resist the Recall Trap
SIGNAL_FLOOR = 0.15        # The absolute noise floor.
MIN_FOLDS_FOR_TRUST = 3    # A k-fold average from 1-2 folds is not a reliable estimate
EPSILON = 1e-9
# Flag-all guard: the certified threshold's precision must beat the base rate
LIFT_FLOOR = 1.5

# ---------------------------------------------------------
# DEPENDENCY GUARD & PATH RESOLUTION
# ---------------------------------------------------------
required_globals = ["TARGET_REPO", "BASE_DIR", "TIME_SPLIT_RATIO"]
for var in required_globals:
    if var not in globals() or globals()[var] is None:
        raise RuntimeError(f"❌ Phase T.3 Dependency Error: '{var}' is missing. Did you run Phase 0 and T.2?")

base_dir_p = Path(BASE_DIR)
TARGET_REPO = str(TARGET_REPO)
TIME_SPLIT_RATIO = float(TIME_SPLIT_RATIO)

T2_TRAIN_CSV_PATH = base_dir_p / f"t2_train_chrono_{int(TIME_SPLIT_RATIO*100)}_{TARGET_REPO}.csv"
CHRONO_THRESHOLDS_PATH = base_dir_p / f"t3_inflection_thresholds_{TARGET_REPO}_chrono_locked.json"
CHRONO_THRESHOLDS_PATH.parent.mkdir(parents=True, exist_ok=True)

if not T2_TRAIN_CSV_PATH.exists():
    raise FileNotFoundError(f"❌ Phase T.3 Error: Could not find {T2_TRAIN_CSV_PATH.name}. You must run Phase T.2 first!")

df_train_t3 = pd.read_csv(T2_TRAIN_CSV_PATH)

# Required-column guardrails
required_cols = {
    "commit_sha", "commit_time", "file_basename", "method_name",
    "is_refactored", "granular_rule", "metric_value", "Calculated_MethodCount"
}
missing_cols = required_cols - set(df_train_t3.columns)
if missing_cols:
    raise RuntimeError(f"❌ Phase T.3 Error: Missing required columns: {sorted(missing_cols)}")

# Strict chronological sliding window at the RELEASE level
df_train_t3 = df_train_t3.sort_values("commit_time").reset_index(drop=True)

train_basenames_set = set(df_train_t3["file_basename"].dropna().unique())
print(f"📊 Loaded Temporal Training Set (The Past) with {len(train_basenames_set)} canonical entities.")

# ---------------------------------------------------------
# 2. METHOD COUNT FOR TooManyMethods — guarded
# ---------------------------------------------------------
if "Calculated_MethodCount" not in df_train_t3.columns:
    raise RuntimeError("❌ Phase T.3: 'Calculated_MethodCount' missing — re-run D.2 (it now computes this) then T.2.")


# 3. Generate Inner Temporal Folds (Strict Release-Level TimeSeriesSplit)
unique_shas = df_train_t3.groupby("commit_sha")["commit_time"].min().sort_values().index.tolist()
sha_array = np.array(unique_shas)

# Dynamic fallback for small repositories
chosen_n_splits = 5
for n_splits in [5, 3, 2]:
    if len(unique_shas) >= n_splits + 1:
        tscv = TimeSeriesSplit(n_splits=n_splits)
        chosen_n_splits = n_splits
        break

if 'tscv' not in locals():
    raise RuntimeError("❌ T.3 Error: Insufficient unique training releases. Cannot create TimeSeriesSplit.")

df_train_t3["Chrono_Fold_ID"] = -1

for fold_idx, (train_sha_idx, val_sha_idx) in enumerate(tscv.split(sha_array), 1):
    val_shas = set(sha_array[val_sha_idx])
    df_train_t3.loc[df_train_t3["commit_sha"].isin(val_shas), "Chrono_Fold_ID"] = fold_idx

# ---------------------------------------------------------
# THE K-FOLD CALIBRATION ENGINE (TEMPORAL EDITION)
# ---------------------------------------------------------
fold_ids = sorted([f for f in df_train_t3["Chrono_Fold_ID"].dropna().unique() if int(f) > 0])
if not fold_ids:
    raise RuntimeError("❌ Phase T.3 Error: No Chrono_Fold_ID values found after split generation.")

print(f"\n📈 Executing {len(fold_ids)}-Fold Cross-Validation Tournament (Chronological; n_splits={chosen_n_splits})...")

sweep_percentiles = [0.05, 0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85, 0.90, 0.95]

rules_to_calc = [
    "NcssCount_Class", "CyclomaticComplexity_Class", "ExcessivePublicCount",
    "CouplingBetweenObjects", "ExcessiveImports", "NcssCount_Method",
    "CyclomaticComplexity_Method", "TooManyMethods"
]

all_rules_in_data = set(df_train_t3["granular_rule"].dropna().unique())
uncalibrated = all_rules_in_data - set(rules_to_calc)
if uncalibrated:
    print(f"⚠️ Note: {len(uncalibrated)} raw PMD rules skipped: {uncalibrated}")

rule_performance = {rule: {ptile: [] for ptile in sweep_percentiles} for rule in rules_to_calc}

for val_fold in fold_ids:
    print(f"   ⚙️ Processing Temporal Fold {val_fold} as Validation...")

    train_k = df_train_t3[
        (df_train_t3["Chrono_Fold_ID"] == -1) |
        (df_train_t3["Chrono_Fold_ID"] < val_fold)
    ].copy()

    val_k = df_train_t3[df_train_t3["Chrono_Fold_ID"] == val_fold].copy()

    for rule in rules_to_calc:
        if rule == "TooManyMethods":
            # Per-CLASS metric: ONE row per class, God-Class label only
            GOD_CLASS_RULES = ["NcssCount_Class", "CyclomaticComplexity_Class",
                               "ExcessivePublicCount", "TooManyMethods"]
            k_train_data = (train_k[train_k["granular_rule"].isin(GOD_CLASS_RULES)]
                            .drop_duplicates(subset=["commit_sha", "file_basename"]))
            k_val_data = (val_k[val_k["granular_rule"].isin(GOD_CLASS_RULES)]
                          .drop_duplicates(subset=["commit_sha", "file_basename"]).copy())
            k_val_data["val"] = k_val_data["Calculated_MethodCount"]
            metric_col = "Calculated_MethodCount"
        else:
            k_train_data = train_k[train_k["granular_rule"] == rule]
            k_val_data = val_k[val_k["granular_rule"] == rule].copy()
            k_val_data["val"] = k_val_data["metric_value"]
            metric_col = "metric_value"

        if k_val_data.empty:
            continue

        bp = k_train_data[k_train_data["is_refactored"] == 1][metric_col]
        if len(bp) < MIN_POS_SUPPORT_TRAIN:
            continue

        quantiles = bp.quantile(sweep_percentiles, interpolation="linear").to_dict()

        v_actual_int = k_val_data["is_refactored"].astype(int)
        if v_actual_int.sum() < MIN_POS_SUPPORT_VAL:
            continue

        for ptile, thresh in quantiles.items():
            ithresh = int(math.ceil(thresh))
            v_pred = (k_val_data["val"] >= ithresh).astype(int)

            p, r, f1, _ = precision_recall_fscore_support(
                v_actual_int, v_pred, average="binary", zero_division=0
            )

            f0_5 = ((1 + 0.5**2) * (p * r)) / ((0.5**2 * p) + r + 1e-9)

            rule_performance[rule][ptile].append({
                "thresh": ithresh, "p": p, "r": r, "f1": f1, "f0_5": f0_5,
                "base": float(v_actual_int.mean())  # base rate = precision of flag-all
            })

# ---------------------------------------------------------
# PRINT THE SWEEP MATRICES
# ---------------------------------------------------------
matrix_rows_f1 = []
matrix_rows_f0_5 = []

for rule in rules_to_calc:
    row_f1 = {"Rule": rule}
    row_f0_5 = {"Rule": rule}
    for ptile in sweep_percentiles:
        results = rule_performance[rule][ptile]
        if results:
            avg_p = sum(res["p"] for res in results) / len(results)
            avg_r = sum(res["r"] for res in results) / len(results)
            avg_f1 = sum(res["f1"] for res in results) / len(results)
            avg_f0_5 = sum(res["f0_5"] for res in results) / len(results)
            med_thresh = int(math.ceil(statistics.median(res["thresh"] for res in results)))

            row_f1[f"{int(ptile*100)}th"] = f"{med_thresh} (P:{avg_p:.2f}|R:{avg_r:.2f}|F1:{avg_f1:.2f})"
            row_f0_5[f"{int(ptile*100)}th"] = f"{med_thresh} (P:{avg_p:.2f}|R:{avg_r:.2f}|F0.5:{avg_f0_5:.2f})"
        else:
            row_f1[f"{int(ptile*100)}th"] = "N/A"
            row_f0_5[f"{int(ptile*100)}th"] = "N/A"

    matrix_rows_f1.append(row_f1)
    matrix_rows_f0_5.append(row_f0_5)

print("\n🔍 Standard F1 Sweep Matrix [ Int Threshold (P | R | F1) ]:")
print(pd.DataFrame(matrix_rows_f1).to_markdown(index=False))

print("\n🎯 Precision-Optimized Sweep Matrix [ Int Threshold (P | R | F0.5) ]:")
print(pd.DataFrame(matrix_rows_f0_5).to_markdown(index=False))

# ---------------------------------------------------------
# AGGREGATE & LOCK DUAL THRESHOLDS (STRICT ARGMAX + STABILITY + TRUST)
# ---------------------------------------------------------
locked_f1_thresholds = {}
locked_f0_5_thresholds = {}

for rule in rules_to_calc:
    # 1. Track Strict Argmax (The New Default)
    argmax_avg_f1, argmax_ptile_f1, argmax_thresh_f1 = -1.0, None, None
    argmax_avg_f0_5, argmax_ptile_f0_5, argmax_thresh_f0_5 = -1.0, None, None
    # Precision & base rate at the winning percentile (for the flag-all lift guard)
    argmax_p_f1, argmax_base_f1 = 0.0, 0.0
    argmax_p_f0_5, argmax_base_f0_5 = 0.0, 0.0

    band_avg_f1, band_ptile_f1, band_thresh_f1 = -1.0, None, None
    band_avg_f0_5, band_ptile_f0_5, band_thresh_f0_5 = -1.0, None, None

    winning_results_count_f1 = 0
    winning_results_count_f0_5 = 0

    for ptile in sweep_percentiles:
        results = rule_performance[rule][ptile]
        if not results:
            continue

        avg_f1 = sum(res["f1"] for res in results) / len(results)
        avg_f0_5 = sum(res["f0_5"] for res in results) / len(results)
        avg_p = sum(res["p"] for res in results) / len(results)
        avg_base = sum(res.get("base", 0.0) for res in results) / len(results)
        median_thresh = int(math.ceil(statistics.median(res["thresh"] for res in results)))

        # ----- Strict Argmax Logic (New Default) -----
        if avg_f1 > argmax_avg_f1 + EPSILON:
            argmax_avg_f1 = avg_f1
            argmax_ptile_f1 = f"{int(ptile*100)}th"
            argmax_thresh_f1 = median_thresh
            winning_results_count_f1 = len(results)
            argmax_p_f1 = avg_p
            argmax_base_f1 = avg_base

        if avg_f0_5 > argmax_avg_f0_5 + EPSILON:
            argmax_avg_f0_5 = avg_f0_5
            argmax_ptile_f0_5 = f"{int(ptile*100)}th"
            argmax_thresh_f0_5 = median_thresh
            winning_results_count_f0_5 = len(results)
            argmax_p_f0_5 = avg_p
            argmax_base_f0_5 = avg_base

        within_band_f1 = avg_f1 >= (band_avg_f1 - F1_TOLERANCE)
        prev_band_f1 = band_thresh_f1 if band_thresh_f1 is not None else float("-inf")

        if (avg_f1 > band_avg_f1 + EPSILON) or (within_band_f1 and median_thresh > prev_band_f1):
            band_avg_f1 = avg_f1
            band_ptile_f1 = f"{int(ptile*100)}th"
            band_thresh_f1 = median_thresh

        within_band_f0_5 = avg_f0_5 >= (band_avg_f0_5 - F1_TOLERANCE)
        prev_band_f0_5 = band_thresh_f0_5 if band_thresh_f0_5 is not None else float("-inf")

        if (avg_f0_5 > band_avg_f0_5 + EPSILON) or (within_band_f0_5 and median_thresh > prev_band_f0_5):
            band_avg_f0_5 = avg_f0_5
            band_ptile_f0_5 = f"{int(ptile*100)}th"
            band_thresh_f0_5 = median_thresh

    # ----- 3. Determine Stability & Trustworthiness -----
    is_stable_f1 = False
    if argmax_thresh_f1 is not None and band_thresh_f1 is not None:
        max_t = max(argmax_thresh_f1, band_thresh_f1)
        min_t = max(min(argmax_thresh_f1, band_thresh_f1), 1)
        is_stable_f1 = (max_t / min_t) <= 2.0

    # F1 Trust Logic
    signal_ok_f1 = (argmax_avg_f1 != -1.0) and (argmax_avg_f1 >= SIGNAL_FLOOR)
    enough_folds_f1 = winning_results_count_f1 >= MIN_FOLDS_FOR_TRUST
    precision_lift_f1 = (argmax_p_f1 / argmax_base_f1) if argmax_base_f1 > EPSILON else 0.0
    lift_ok_f1 = precision_lift_f1 >= LIFT_FLOOR  # rejects flag-all (lift ≈ 1.0)
    trustworthy_f1 = is_stable_f1 and signal_ok_f1 and enough_folds_f1 and lift_ok_f1

    is_stable_f0_5 = False
    if argmax_thresh_f0_5 is not None and band_thresh_f0_5 is not None:
        max_t = max(argmax_thresh_f0_5, band_thresh_f0_5)
        min_t = max(min(argmax_thresh_f0_5, band_thresh_f0_5), 1)
        is_stable_f0_5 = (max_t / min_t) <= 2.0

    # F0.5 Trust Logic
    signal_ok_f0_5 = (argmax_avg_f0_5 != -1.0) and (argmax_avg_f0_5 >= SIGNAL_FLOOR)
    enough_folds_f0_5 = winning_results_count_f0_5 >= MIN_FOLDS_FOR_TRUST
    precision_lift_f0_5 = (argmax_p_f0_5 / argmax_base_f0_5) if argmax_base_f0_5 > EPSILON else 0.0
    lift_ok_f0_5 = precision_lift_f0_5 >= LIFT_FLOOR
    trustworthy_f0_5 = is_stable_f0_5 and signal_ok_f0_5 and enough_folds_f0_5 and lift_ok_f0_5

    locked_f1_thresholds[rule] = {
        "calibrated": argmax_thresh_f1 is not None,
        "threshold_stable": is_stable_f1,
        "trustworthy": trustworthy_f1,
        "percentile": argmax_ptile_f1 if argmax_ptile_f1 else "Default",
        "threshold_value": int(argmax_thresh_f1) if argmax_thresh_f1 is not None else None,
        "band_divergence": int(band_thresh_f1) if band_thresh_f1 is not None else None,
        "kfold_avg_f1": round(argmax_avg_f1, 4) if argmax_avg_f1 != -1.0 else None,
        "precision_lift": round(precision_lift_f1, 2),
        "base_rate": round(argmax_base_f1, 4),
        "folds_used": winning_results_count_f1
    }

    locked_f0_5_thresholds[rule] = {
        "calibrated": argmax_thresh_f0_5 is not None,
        "threshold_stable": is_stable_f0_5,
        "trustworthy": trustworthy_f0_5,
        "percentile": argmax_ptile_f0_5 if argmax_ptile_f0_5 else "Default",
        "threshold_value": int(argmax_thresh_f0_5) if argmax_thresh_f0_5 is not None else None,
        "band_divergence": int(band_thresh_f0_5) if band_thresh_f0_5 is not None else None,
        "kfold_avg_f0_5": round(argmax_avg_f0_5, 4) if argmax_avg_f0_5 != -1.0 else None,
        "precision_lift": round(precision_lift_f0_5, 2),
        "base_rate": round(argmax_base_f0_5, 4),
        "folds_used": winning_results_count_f0_5
    }

# ---------------------------------------------------------
# PRINT LOCKED THRESHOLDS & SAVE TO JSON
# ---------------------------------------------------------
print("\n🏆 F1-Optimized Chronological Thresholds:")
print(pd.DataFrame(locked_f1_thresholds).T.to_markdown())

print("\n🎯 Precision-Optimized (F0.5) Chronological Thresholds:")
print(pd.DataFrame(locked_f0_5_thresholds).T.to_markdown())

final_json_export = locked_f1_thresholds.copy()
final_json_export["_precision_optimized_f0_5"] = locked_f0_5_thresholds

final_json_export["_meta"] = {
    "chosen_n_splits": chosen_n_splits,
    "min_pos_support_val": MIN_POS_SUPPORT_VAL,
    "min_pos_support_train": MIN_POS_SUPPORT_TRAIN,
    "min_folds_for_trust": MIN_FOLDS_FOR_TRUST,
    "lift_floor": LIFT_FLOOR,
    "f1_tolerance_band": F1_TOLERANCE,
    "signal_floor": SIGNAL_FLOOR,
    "quantile_interpolation": "linear"
}

with open(CHRONO_THRESHOLDS_PATH, "w", encoding="utf-8") as f:
    json.dump(final_json_export, f, indent=4)

print(f"\n💾 Saved Dual Chronological locked thresholds to disk: {CHRONO_THRESHOLDS_PATH.name}")
print("🏁 Phase T.3 Complete.")

## Phase 4: Blind Predictability Evaluations
This phase executes the final, blind exams. It applies the JSON thresholds derived in Phase 3 against their respective holdout sets to definitively measure their absolute and relative improvements over factory defaults.

### ⚙️ Phase B.4: Spatial Baseline Evaluation
**Objective:** Blindly evaluate the spatial thresholds against the spatial holdout.
**Logic Flow:**
* **Data Binding:** Load the unseen B.2 spatial test matrix and the locked B.3 spatial JSON thresholds.
* **Trust Fallback:** For each rule, check the JSON calibration metadata. If the threshold was flagged `trustworthy: False`, trigger a strict fallback to factory PMD defaults.
* **Scoring Engine:** Evaluate the test set using a full suite of metrics (Precision, Recall, F1, F0.5, MCC).
* **Anchor Comparisons:** Calculate PMD default performance and a Naive Median threshold performance to serve as comparison anchors.
* **Evaluation Matrix:** Output the Master Spatial Evaluation Matrix, highlighting absolute improvements and flagging severe performance degradations.

In [ ]:
import math
import pandas as pd
import json
import warnings
from pathlib import Path
from sklearn.metrics import precision_recall_fscore_support, matthews_corrcoef, average_precision_score

def get_stats(y_true, y_pred, scores):
    y_true_int = y_true.astype(int)
    y_pred_int = y_pred.astype(int)

    p, r, f1, _ = precision_recall_fscore_support(
        y_true_int, y_pred_int, average='binary', zero_division=0
    )
    # Explicit F0.5 Calculation
    f0_5 = ((1 + 0.5**2) * (p * r)) / ((0.5**2 * p) + r + 1e-9)

    mcc = matthews_corrcoef(y_true_int, y_pred_int) if len(set(y_true_int)) > 1 and len(set(y_pred_int)) > 1 else 0.0
    try:
        auc_pr = average_precision_score(y_true_int, scores)
    except (ValueError, TypeError):
        auc_pr = 0.0
    return p, r, f1, f0_5, mcc, auc_pr, int(y_true_int.sum())

# Suppress warnings for a clean academic report
warnings.simplefilter(action='ignore', category=FutureWarning)

# Guard for pandas-version-specific option availability
try:
    pd.set_option('future.no_silent_downcasting', True)
except Exception:
    pass

print("🚀 Starting Phase B.4: Blind Imminent Predictability Evaluation (Trust-Aware Edition)")

# ---------------------------------------------------------
# DEPENDENCY GUARD & PATHS
# ---------------------------------------------------------
required_globals = ["TARGET_REPO", "BASE_DIR"]
for var in required_globals:
    if var not in globals() or globals()[var] is None:
        raise RuntimeError(f"❌ Phase B.4 Dependency Error: '{var}' is missing. Did you run Phase 0?")

base_dir_p = Path(BASE_DIR)
TARGET_REPO = str(TARGET_REPO)

# Tie directly into the established B.2 and B.3 architecture
TEST_CSV_PATH = base_dir_p / f"b2_test_spatial_20_{TARGET_REPO}.csv"
TRAIN_CSV_PATH = base_dir_p / f"b2_train_spatial_80_{TARGET_REPO}.csv"
LOCKED_THRESHOLDS_PATH = base_dir_p / f"b3_inflection_thresholds_{TARGET_REPO}_locked.json"
EVAL_CSV_PATH = base_dir_p / f"b4_evaluation_matrix_{TARGET_REPO}.csv"
EVAL_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

# File guards
for p in [TEST_CSV_PATH, TRAIN_CSV_PATH, LOCKED_THRESHOLDS_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"❌ Phase B.4 Error: Required file not found: {p.name}")

# 2. Load the Data & Locked Thresholds
df_test = pd.read_csv(TEST_CSV_PATH)
df_train = pd.read_csv(TRAIN_CSV_PATH)

# Required-column guards
required_cols = {"commit_sha", "file_basename", "method_name", "granular_rule", "metric_value", "is_refactored", "Calculated_MethodCount"}
missing_test = required_cols - set(df_test.columns)
missing_train = required_cols - set(df_train.columns)

if missing_test:
    raise RuntimeError(f"❌ Phase B.4 Error: Test CSV missing required columns: {sorted(missing_test)}")
if missing_train:
    raise RuntimeError(f"❌ Phase B.4 Error: Train CSV missing required columns: {sorted(missing_train)}")

test_basenames = set(df_test["file_basename"].dropna().unique())

try:
    with open(LOCKED_THRESHOLDS_PATH, "r", encoding="utf-8") as f:
        locked_thresholds = json.load(f)
    print("✅ Thresholds strictly synchronized from disk.")
except Exception as e:
    raise RuntimeError(f"❌ Phase B.4 Error: Could not load locked thresholds from disk: {e}")

if not locked_thresholds:
    raise RuntimeError("❌ Phase B.4 Error: Locked thresholds file is empty. Run Phase B.3 first.")

pmd_defaults = {
    "NcssCount_Class": 1500, "NcssCount_Method": 60,
    "CyclomaticComplexity_Class": 80, "CyclomaticComplexity_Method": 10,
    "ExcessivePublicCount": 45, "CouplingBetweenObjects": 20,
    "ExcessiveImports": 30, "TooManyMethods": 10,
    "ExcessiveParameterList": 10
}

print(f"📊 Final Exam Loaded: Evaluating {len(test_basenames)} completely unseen canonical entities.")

# 3. Method count for TooManyMethods comes from D.2 (Calculated_MethodCount),
#    carried through the split CSVs — do NOT recompute it here. It must match
#    B.3's calibration metric exactly.

# 4. The Blind Evaluation Engine
f1_records = []

# THE FIX: Explicitly exclude EPL due to PMD mapping limitations (Tooling Gap, not Noise)
CALIBRATION_EXCLUDED_RULES = {"ExcessiveParameterList"}

# Ignore any metadata keys starting with "_" AND exclude the skipped rules
rules_to_evaluate = [
    k for k in locked_thresholds.keys()
    if not k.startswith("_") and k not in CALIBRATION_EXCLUDED_RULES
]

for rule in rules_to_evaluate:
    if rule == "TooManyMethods":
        # MUST match B.3's calibration selection: one row per class, God-Class
        # label only, engineered method count as the metric.
        GOD_CLASS_RULES = ["NcssCount_Class", "CyclomaticComplexity_Class",
                           "ExcessivePublicCount", "TooManyMethods"]
        t_data = (df_test[df_test["granular_rule"].isin(GOD_CLASS_RULES)]
                  .drop_duplicates(subset=["commit_sha", "file_basename"]).copy())
        t_data["val"] = t_data["Calculated_MethodCount"]
        train_val = (df_train[df_train["granular_rule"].isin(GOD_CLASS_RULES)]
                     .drop_duplicates(subset=["commit_sha", "file_basename"])["Calculated_MethodCount"])
    else:
        t_data = df_test[df_test["granular_rule"] == rule].copy()
        t_data["val"] = t_data["metric_value"]
        train_val = df_train[df_train["granular_rule"] == rule]["metric_value"]

    if t_data.empty:
        continue

    t_actual = t_data["is_refactored"].astype(int)

    if t_actual.sum() == 0:
        print(f"⚠️  Skipping {rule}: No refactoring events found in test set.")
        continue

    # Grab both F1 and F0.5 dictionaries
    rule_data_f1 = locked_thresholds.get(rule, {})
    rule_data_f05 = locked_thresholds.get("_precision_optimized_f0_5", {}).get(rule, {})

    # Extract Trust & Calibrated Flags
    trust_f1 = rule_data_f1.get("trustworthy", False)
    trust_f05 = rule_data_f05.get("trustworthy", False)

    raw_thresh_f1 = rule_data_f1.get("threshold_value")
    raw_thresh_f05 = rule_data_f05.get("threshold_value")

    pmd_thresh = pmd_defaults.get(rule, 9999)
    naive_thresh = int(math.ceil(train_val.median())) if not train_val.empty else 9999

    # Safely handle None thresholds (Fallback to PMD if not trustworthy)
    eval_thresh_f1 = raw_thresh_f1 if (trust_f1 and raw_thresh_f1 is not None) else pmd_thresh
    eval_thresh_f05 = raw_thresh_f05 if (trust_f05 and raw_thresh_f05 is not None) else pmd_thresh

    # Unpack the output from get_stats
    pmd_p, pmd_r, pmd_f1, pmd_f05, pmd_mcc, _, support = get_stats(
        t_actual, (t_data["val"] >= pmd_thresh), t_data["val"]
    )
    naive_p, naive_r, naive_f1, naive_f05, naive_mcc, _, _ = get_stats(
        t_actual, (t_data["val"] >= naive_thresh), t_data["val"]
    )

    # Evaluate F1-Optimized Custom Threshold
    c_f1_p, c_f1_r, c_f1_f1, c_f1_f05, c_f1_mcc, auc_pr, _ = get_stats(
        t_actual, (t_data["val"] >= eval_thresh_f1), t_data["val"]
    )

    # Evaluate Precision-Optimized (F0.5) Custom Threshold
    c_f05_p, c_f05_r, c_f05_f1, c_f05_f05, c_f05_mcc, _, _ = get_stats(
        t_actual, (t_data["val"] >= eval_thresh_f05), t_data["val"]
    )

    f1_records.append({
        "Metric": rule, "Support": support,
        "PMD_Thresh": pmd_thresh,
        "Trust_F1": trust_f1, "Thresh_F1": eval_thresh_f1,
        "Trust_F05": trust_f05, "Thresh_F05": eval_thresh_f05,
        "Naive_Thresh": naive_thresh,
        "Hist_F1": rule_data_f1.get("kfold_avg_f1", 0.0) or 0.0,
        "Hist_F05": rule_data_f05.get("kfold_avg_f0_5", 0.0) or 0.0,
        "Cust_F1_P": c_f1_p, "Cust_F1_R": c_f1_r, "Cust_F1_F1": c_f1_f1, "Cust_F1_F05": c_f1_f05,
        "Cust_F05_P": c_f05_p, "Cust_F05_R": c_f05_r, "Cust_F05_F1": c_f05_f1, "Cust_F05_F05": c_f05_f05,
        "PMD_P": pmd_p, "PMD_R": pmd_r, "PMD_F1": pmd_f1, "PMD_F05": pmd_f05,
        "Naive_P": naive_p, "Naive_R": naive_r, "Naive_F1": naive_f1, "Naive_F05": naive_f05,
        "PMD_MCC": pmd_mcc, "Naive_MCC": naive_mcc, "Cust_F1_MCC": c_f1_mcc, "Cust_F05_MCC": c_f05_mcc,
        "AUC_PR": auc_pr
    })

# 5. Presentation & Save
df_final = pd.DataFrame(f1_records)
print("\n    MASTER BLIND EVALUATION MATRIX (Test Set: 20%)")

if not df_final.empty:
    # --- DRIFT LOGIC ---
    def evaluate_drift_f1(row):
        if not row["Trust_F1"]: return "➖ Uncalibrated Noise"
        cust = float(row["Cust_F1_F1"])
        pmd = float(row["PMD_F1"])
        hist = float(row["Hist_F1"])
        if cust < pmd: return "🚨 Severe (PMD Wins)"
        elif hist > 0 and (hist - cust) >= 0.05: return "⚠️ Degradation"
        return "✅ Stable"

    def evaluate_drift_f05(row):
        if not row["Trust_F05"]: return "➖ Uncalibrated Noise"
        cust = float(row["Cust_F05_F05"])
        pmd = float(row["PMD_F05"])
        hist = float(row["Hist_F05"])
        if cust < pmd: return "🚨 Severe (PMD Wins)"
        elif hist > 0 and (hist - cust) >= 0.05: return "⚠️ Degradation"
        return "✅ Stable"

    df_final["Drift_Alert_F1"] = df_final.apply(evaluate_drift_f1, axis=1)
    df_final["Drift_Alert_F05"] = df_final.apply(evaluate_drift_f05, axis=1)

    # --- FORMATTING FOR DISPLAY ---
    df_display = df_final.copy()

    # Annotate untrustworthy thresholds with a warning flag in the table
    df_display["Display_Thresh_F1"] = df_display.apply(
        lambda r: f"{r['Thresh_F1']}" if r["Trust_F1"] else f"⚠️ {r['Thresh_F1']} (Fallback)", axis=1
    )
    df_display["Display_Thresh_F05"] = df_display.apply(
        lambda r: f"{r['Thresh_F05']}" if r["Trust_F05"] else f"⚠️ {r['Thresh_F05']} (Fallback)", axis=1
    )

    float_cols = [
        "Hist_F1", "Hist_F05", "Cust_F1_F1", "Cust_F1_F05", "Cust_F05_F1", "Cust_F05_F05",
        "PMD_F1", "PMD_F05", "Naive_F1", "Naive_F05"
    ]
    for c in float_cols:
        df_display[c] = df_display[c].apply(lambda x: f"{float(x):.3f}")

    # --- TABLE 1: F1 Evaluation ---
    df_f1 = df_display[["Metric", "Support", "PMD_Thresh", "Naive_Thresh", "Display_Thresh_F1", "Hist_F1", "Cust_F1_F1", "PMD_F1", "Naive_F1", "Drift_Alert_F1"]].copy()
    df_f1.rename(columns={"Display_Thresh_F1": "Cust_Thresh", "Cust_F1_F1": "Cust_F1"}, inplace=True)

    # --- TABLE 2: F0.5 Evaluation ---
    df_f05 = df_display[["Metric", "Support", "PMD_Thresh", "Naive_Thresh", "Display_Thresh_F05", "Hist_F05", "Cust_F05_F05", "PMD_F05", "Naive_F05", "Drift_Alert_F05"]].copy()
    df_f05.rename(columns={"Display_Thresh_F05": "Cust_Thresh", "Cust_F05_F05": "Cust_F05"}, inplace=True)

    print("\n📊 TABLE 1: F1-Optimized Thresholds (Evaluated on Blind F1 Score)")
    print(df_f1.to_markdown(index=False))

    print("\n🎯 TABLE 2: Precision-Optimized Thresholds (Evaluated on Blind F0.5 Score)")
    print(df_f05.to_markdown(index=False))

    df_final.to_csv(EVAL_CSV_PATH, index=False)
    print(f"\n💾 Saved detailed evaluation matrix to: {EVAL_CSV_PATH.name}")

    # --- THE HEADLINE SUMMARY (FILTERED ONLY TO TRUSTWORTHY RULES) ---
    trust_df_f1 = df_final[df_final["Trust_F1"] == True]
    total_support_f1 = trust_df_f1["Support"].sum()

    trust_df_f05 = df_final[df_final["Trust_F05"] == True]
    total_support_f05 = trust_df_f05["Support"].sum()

    print("\n" + "=" * 65)
    print("🏆 FINAL BASELINE EVALUATION SUMMARY (Trustworthy Rules Only)")
    print("=" * 65)
    print(f"   Test Entities (Strictly Blind):   {len(test_basenames)}")
    print("-" * 65)

    if total_support_f1 > 0:
        avg_pmd_f1 = (trust_df_f1["PMD_F1"] * trust_df_f1["Support"]).sum() / total_support_f1
        avg_naive_f1 = (trust_df_f1["Naive_F1"] * trust_df_f1["Support"]).sum() / total_support_f1
        avg_cust_f1 = (trust_df_f1["Cust_F1_F1"] * trust_df_f1["Support"]).sum() / total_support_f1
        abs_improvement_f1 = avg_cust_f1 - avg_pmd_f1
        rel_improvement_f1 = (abs_improvement_f1 / avg_pmd_f1 * 100) if avg_pmd_f1 > 0 else 0

        print("   [ TRACK 1: Standard F1 Score ]")
        print(f"   Rules Calibrated Successfully:   {len(trust_df_f1)}")
        print(f"   Weighted PMD:                    {avg_pmd_f1:.3f}")
        print(f"   Weighted Naive:                  {avg_naive_f1:.3f}")
        print(f"   Weighted Custom:                 {avg_cust_f1:.3f}")
        print(f"   Absolute Improvement:          +{abs_improvement_f1:.3f}")
        print(f"   Relative Improvement:          +{rel_improvement_f1:.1f}%" if rel_improvement_f1 > 0 else f"   Relative Impact:               {rel_improvement_f1:.1f}%")
    else:
        print("   [ TRACK 1: Standard F1 Score ]")
        print("   ⚠️ No rules met the Trustworthy criteria. Cannot compute F1 averages.")

    print("-" * 65)

    if total_support_f05 > 0:
        avg_pmd_f05 = (trust_df_f05["PMD_F05"] * trust_df_f05["Support"]).sum() / total_support_f05
        avg_naive_f05 = (trust_df_f05["Naive_F05"] * trust_df_f05["Support"]).sum() / total_support_f05
        avg_cust_f05 = (trust_df_f05["Cust_F05_F05"] * trust_df_f05["Support"]).sum() / total_support_f05
        abs_improvement_f05 = avg_cust_f05 - avg_pmd_f05
        rel_improvement_f05 = (abs_improvement_f05 / avg_pmd_f05 * 100) if avg_pmd_f05 > 0 else 0

        print("   [ TRACK 2: Precision-Weighted F0.5 Score ]")
        print(f"   Rules Calibrated Successfully:   {len(trust_df_f05)}")
        print(f"   Weighted PMD:                    {avg_pmd_f05:.3f}")
        print(f"   Weighted Naive:                  {avg_naive_f05:.3f}")
        print(f"   Weighted Custom:                 {avg_cust_f05:.3f}")
        print(f"   Absolute Improvement:          +{abs_improvement_f05:.3f}")
        print(f"   Relative Improvement:          +{rel_improvement_f05:.1f}%" if rel_improvement_f05 > 0 else f"   Relative Impact:               {rel_improvement_f05:.1f}%")
    else:
        print("   [ TRACK 2: Precision-Weighted F0.5 Score ]")
        print("   ⚠️ No rules met the Trustworthy criteria. Cannot compute F0.5 averages.")

    print("=" * 65)
else:
    print("⚠️ No valid evaluation records generated. Ensure the Test Set has positive signals.")

print("🏁 Phase B.4 Complete. The threshold hypotheses have been blindly evaluated.")

### ⚙️ Phase E.4: Scarcity Predictability Evaluation
**Objective:** Evaluate threshold survival under data scarcity.
**Logic Flow:**
* **Data Binding:** Load the shared B.2 test matrix and the E.3 scarcity-derived JSON thresholds.
* **Trust Fallback:** Apply the blind evaluation constraints, defaulting to PMD rules if the scarcity calibration failed to isolate a statistically trustworthy signal.
* **Delta Calculation:** Evaluate the test set to determine if data starvation caused a previously stable spatial threshold to collapse into uncalibrated noise.
* **Evaluation Matrix:** Output the Master Scarcity Evaluation Matrix to quantify the impact of limited data volume.

In [ ]:
import math
import pandas as pd
import json
import warnings
from pathlib import Path
from sklearn.metrics import precision_recall_fscore_support, matthews_corrcoef, average_precision_score

def get_stats(y_true, y_pred, scores):
    y_true_int = y_true.astype(int)
    y_pred_int = y_pred.astype(int)

    p, r, f1, _ = precision_recall_fscore_support(
        y_true_int, y_pred_int, average='binary', zero_division=0
    )
    f0_5 = ((1 + 0.5**2) * (p * r)) / ((0.5**2 * p) + r + 1e-9)
    mcc = matthews_corrcoef(y_true_int, y_pred_int) if len(set(y_true_int)) > 1 and len(set(y_pred_int)) > 1 else 0.0
    try:
        auc_pr = average_precision_score(y_true_int, scores)
    except (ValueError, TypeError):
        auc_pr = 0.0
    return p, r, f1, f0_5, mcc, auc_pr, int(y_true_int.sum())

warnings.simplefilter(action='ignore', category=FutureWarning)
try:
    pd.set_option('future.no_silent_downcasting', True)
except Exception:
    pass

print("🚀 Starting Phase E.4: Scarcity Predictability Evaluation (Trust-Aware)")

# ---------------------------------------------------------
# DEPENDENCY GUARD & PATHS
# ---------------------------------------------------------
required_globals = ["TARGET_REPO", "BASE_DIR"]
for var in required_globals:
    if var not in globals() or globals()[var] is None:
        raise RuntimeError(f"❌ Phase E.4 Dependency Error: '{var}' is missing. Did you run Phase 0?")

base_dir_p = Path(BASE_DIR)
TARGET_REPO = str(TARGET_REPO)

# E.4 STRICT LOCK: Bind directly to the established Spatial/Scarcity pipeline tracks
TEST_CSV_PATH = base_dir_p / f"b2_test_spatial_20_{TARGET_REPO}.csv"
TRAIN_CSV_PATH = base_dir_p / f"e2_train_scarcity_{TARGET_REPO}.csv"
SCARCITY_THRESHOLDS_PATH = base_dir_p / f"e3_inflection_thresholds_{TARGET_REPO}_scarcity.json"
EVAL_CSV_PATH = base_dir_p / f"e4_evaluation_matrix_scarcity_{TARGET_REPO}.csv"

for p in [TEST_CSV_PATH, TRAIN_CSV_PATH, SCARCITY_THRESHOLDS_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"❌ Phase E.4 Error: Required file not found: {p.name}")

df_test = pd.read_csv(TEST_CSV_PATH)
df_train = pd.read_csv(TRAIN_CSV_PATH)

# Required-column guards
required_cols = {"commit_sha", "file_basename", "method_name", "granular_rule", "metric_value", "is_refactored", "Calculated_MethodCount"}
missing_test = required_cols - set(df_test.columns)
missing_train = required_cols - set(df_train.columns)

if missing_test:
    raise RuntimeError(f"❌ Phase E.4 Error: Test CSV missing required columns: {sorted(missing_test)}")
if missing_train:
    raise RuntimeError(f"❌ Phase E.4 Error: Train CSV missing required columns: {sorted(missing_train)}")

test_basenames = set(df_test["file_basename"].dropna().unique())

with open(SCARCITY_THRESHOLDS_PATH, "r", encoding="utf-8") as f:
    locked_thresholds = json.load(f)

pmd_defaults = {
    "NcssCount_Class": 1500, "NcssCount_Method": 60,
    "CyclomaticComplexity_Class": 80, "CyclomaticComplexity_Method": 10,
    "ExcessivePublicCount": 45, "CouplingBetweenObjects": 20,
    "ExcessiveImports": 30, "TooManyMethods": 10,
    "ExcessiveParameterList": 10
}

print(f"📊 Final Scarcity Exam Loaded: Evaluating {len(test_basenames)} completely unseen canonical entities.")

# 4. The Blind Evaluation Engine
f1_records = []

# Explicitly exclude EPL due to PMD mapping limitations (Tooling Gap, not Noise)
CALIBRATION_EXCLUDED_RULES = {"ExcessiveParameterList"}

# Ignore any metadata keys starting with "_" AND exclude the skipped rules
rules_to_evaluate = [
    k for k in locked_thresholds.keys()
    if not k.startswith("_") and k not in CALIBRATION_EXCLUDED_RULES
]

for rule in rules_to_evaluate:
    if rule == "TooManyMethods":
        GOD_CLASS_RULES = ["NcssCount_Class", "CyclomaticComplexity_Class",
                           "ExcessivePublicCount", "TooManyMethods"]
        t_data = (df_test[df_test["granular_rule"].isin(GOD_CLASS_RULES)]
                  .drop_duplicates(subset=["commit_sha", "file_basename"]).copy())
        t_data["val"] = t_data["Calculated_MethodCount"]
        train_val = (df_train[df_train["granular_rule"].isin(GOD_CLASS_RULES)]
                     .drop_duplicates(subset=["commit_sha", "file_basename"])["Calculated_MethodCount"])
    else:
        t_data = df_test[df_test["granular_rule"] == rule].copy()
        t_data["val"] = t_data["metric_value"]
        train_val = df_train[df_train["granular_rule"] == rule]["metric_value"]

    if t_data.empty: continue
    t_actual = t_data["is_refactored"].astype(int)
    if t_actual.sum() == 0: continue

    rule_data_f1 = locked_thresholds.get(rule, {})
    rule_data_f05 = locked_thresholds.get("_precision_optimized_f0_5", {}).get(rule, {})

    trust_f1 = rule_data_f1.get("trustworthy", False)
    trust_f05 = rule_data_f05.get("trustworthy", False)

    raw_thresh_f1 = rule_data_f1.get("threshold_value")
    raw_thresh_f05 = rule_data_f05.get("threshold_value")

    pmd_thresh = pmd_defaults.get(rule, 9999)
    naive_thresh = int(math.ceil(train_val.median())) if not train_val.empty else 9999

    eval_thresh_f1 = raw_thresh_f1 if (trust_f1 and raw_thresh_f1 is not None) else pmd_thresh
    eval_thresh_f05 = raw_thresh_f05 if (trust_f05 and raw_thresh_f05 is not None) else pmd_thresh

    pmd_p, pmd_r, pmd_f1, pmd_f05, pmd_mcc, _, support = get_stats(t_actual, (t_data["val"] >= pmd_thresh), t_data["val"])
    naive_p, naive_r, naive_f1, naive_f05, naive_mcc, _, _ = get_stats(t_actual, (t_data["val"] >= naive_thresh), t_data["val"])
    c_f1_p, c_f1_r, c_f1_f1, c_f1_f05, c_f1_mcc, auc_pr, _ = get_stats(t_actual, (t_data["val"] >= eval_thresh_f1), t_data["val"])
    c_f05_p, c_f05_r, c_f05_f1, c_f05_f05, c_f05_mcc, _, _ = get_stats(t_actual, (t_data["val"] >= eval_thresh_f05), t_data["val"])

    f1_records.append({
        "Metric": rule, "Support": support,
        "PMD_Thresh": pmd_thresh,
        "Trust_F1": trust_f1, "Thresh_F1": eval_thresh_f1,
        "Trust_F05": trust_f05, "Thresh_F05": eval_thresh_f05,
        "Naive_Thresh": naive_thresh,
        "Hist_F1": rule_data_f1.get("kfold_avg_f1", 0.0) or 0.0,
        "Hist_F05": rule_data_f05.get("kfold_avg_f0_5", 0.0) or 0.0,
        "Cust_F1_F1": c_f1_f1, "Cust_F05_F05": c_f05_f05,
        "PMD_F1": pmd_f1, "PMD_F05": pmd_f05,
        "Naive_F1": naive_f1, "Naive_F05": naive_f05
    })

df_final = pd.DataFrame(f1_records)
if not df_final.empty:
    def evaluate_drift_f1(row):
        if not row["Trust_F1"]: return "➖ Uncalibrated Noise"
        cust, pmd, hist = float(row["Cust_F1_F1"]), float(row["PMD_F1"]), float(row["Hist_F1"])
        if cust < pmd: return "🚨 Severe (PMD Wins)"
        elif hist > 0 and (hist - cust) >= 0.05: return "⚠️ Degradation"
        return "✅ Stable"

    def evaluate_drift_f05(row):
        if not row["Trust_F05"]: return "➖ Uncalibrated Noise"
        cust, pmd, hist = float(row["Cust_F05_F05"]), float(row["PMD_F05"]), float(row["Hist_F05"])
        if cust < pmd: return "🚨 Severe (PMD Wins)"
        elif hist > 0 and (hist - cust) >= 0.05: return "⚠️ Degradation"
        return "✅ Stable"

    df_final["Drift_Alert_F1"] = df_final.apply(evaluate_drift_f1, axis=1)
    df_final["Drift_Alert_F05"] = df_final.apply(evaluate_drift_f05, axis=1)

    df_display = df_final.copy()
    df_display["Display_Thresh_F1"] = df_display.apply(lambda r: f"{r['Thresh_F1']}" if r["Trust_F1"] else f"⚠️ {r['Thresh_F1']} (Fallback)", axis=1)
    df_display["Display_Thresh_F05"] = df_display.apply(lambda r: f"{r['Thresh_F05']}" if r["Trust_F05"] else f"⚠️ {r['Thresh_F05']} (Fallback)", axis=1)

    float_cols = ["Hist_F1", "Hist_F05", "Cust_F1_F1", "Cust_F05_F05", "PMD_F1", "PMD_F05", "Naive_F1", "Naive_F05"]
    for c in float_cols: df_display[c] = df_display[c].apply(lambda x: f"{float(x):.3f}")

    df_f1 = df_display[["Metric", "Support", "PMD_Thresh", "Naive_Thresh", "Display_Thresh_F1", "Hist_F1", "Cust_F1_F1", "PMD_F1", "Naive_F1", "Drift_Alert_F1"]].copy()
    df_f1.rename(columns={"Display_Thresh_F1": "Cust_Thresh", "Cust_F1_F1": "Cust_F1"}, inplace=True)

    df_f05 = df_display[["Metric", "Support", "PMD_Thresh", "Naive_Thresh", "Display_Thresh_F05", "Hist_F05", "Cust_F05_F05", "PMD_F05", "Naive_F05", "Drift_Alert_F05"]].copy()
    df_f05.rename(columns={"Display_Thresh_F05": "Cust_Thresh", "Cust_F05_F05": "Cust_F05"}, inplace=True)

    print("\n📊 TABLE 1: F1-Optimized Scarcity Evaluation")
    print(df_f1.to_markdown(index=False))

    print("\n🎯 TABLE 2: Precision-Optimized (F0.5) Scarcity Evaluation")
    print(df_f05.to_markdown(index=False))

    trust_df_f1 = df_final[df_final["Trust_F1"] == True]
    trust_df_f05 = df_final[df_final["Trust_F05"] == True]

    print("\n" + "=" * 65)
    print("🏆 SCARCITY EVALUATION SUMMARY (Trustworthy Rules Only)")
    print("=" * 65)

    if len(trust_df_f1) > 0:
        avg_pmd_f1 = (trust_df_f1["PMD_F1"] * trust_df_f1["Support"]).sum() / trust_df_f1["Support"].sum()
        avg_cust_f1 = (trust_df_f1["Cust_F1_F1"] * trust_df_f1["Support"]).sum() / trust_df_f1["Support"].sum()
        print("   [ TRACK 1: Standard F1 Score ]")
        print(f"   Rules Survived Scarcity:         {len(trust_df_f1)}")
        print(f"   Weighted PMD:                    {avg_pmd_f1:.3f}")
        print(f"   Weighted Custom:                 {avg_cust_f1:.3f}")
    else:
        print("   [ TRACK 1: Standard F1 Score ]")
        print("   ⚠️ No rules met the Trustworthy criteria under scarcity.")

    print("-" * 65)

    if len(trust_df_f05) > 0:
        avg_pmd_f05 = (trust_df_f05["PMD_F05"] * trust_df_f05["Support"]).sum() / trust_df_f05["Support"].sum()
        avg_cust_f05 = (trust_df_f05["Cust_F05_F05"] * trust_df_f05["Support"]).sum() / trust_df_f05["Support"].sum()
        print("   [ TRACK 2: Precision-Weighted F0.5 Score ]")
        print(f"   Rules Survived Scarcity:         {len(trust_df_f05)}")
        print(f"   Weighted PMD:                    {avg_pmd_f05:.3f}")
        print(f"   Weighted Custom:                 {avg_cust_f05:.3f}")
    else:
        print("   [ TRACK 2: Precision-Weighted F0.5 Score ]")
        print("   ⚠️ No rules met the Trustworthy criteria under scarcity.")
    print("=" * 65)

    df_final.to_csv(EVAL_CSV_PATH, index=False)
    print(f"\n💾 Saved scarcity evaluation matrix to: {EVAL_CSV_PATH.name}")
else:
    print("⚠️ No valid evaluation records generated. Ensure the Test Set has positive signals.")

print("🏁 Phase E.4 Complete. Scarcity evaluation finalized.")

### ⚙️ Phase T.4: Chronological Predictability Evaluation
**Objective:** The ultimate test of temporal generalization.
**Logic Flow:**
* **Future Binding:** Load the completely unseen T.2 "Future" timeline matrix and the T.3 chronological JSON thresholds.
* **Trust Fallback:** Rigorously enforce the PMD default fallback for any rule that failed the temporal calibration guardrails.
* **Temporal Evaluation:** Evaluate whether custom thresholds—derived strictly from past data—maintain their predictive power when applied to future, highly evolved architectural states.
* **Evaluation Matrix:** Generate the definitive Chronological Evaluation Matrix to answer the paper's core hypothesis regarding time-travel data leakage.

In [ ]:
import math
import pandas as pd
import json
import warnings
from pathlib import Path
from sklearn.metrics import precision_recall_fscore_support, matthews_corrcoef, average_precision_score

def get_stats(y_true, y_pred, scores):
    y_true_int = y_true.astype(int)
    y_pred_int = y_pred.astype(int)

    p, r, f1, _ = precision_recall_fscore_support(
        y_true_int, y_pred_int, average='binary', zero_division=0
    )
    f0_5 = ((1 + 0.5**2) * (p * r)) / ((0.5**2 * p) + r + 1e-9)
    mcc = matthews_corrcoef(y_true_int, y_pred_int) if len(set(y_true_int)) > 1 and len(set(y_pred_int)) > 1 else 0.0
    try:
        auc_pr = average_precision_score(y_true_int, scores)
    except (ValueError, TypeError):
        auc_pr = 0.0
    return p, r, f1, f0_5, mcc, auc_pr, int(y_true_int.sum())

warnings.simplefilter(action='ignore', category=FutureWarning)
try:
    pd.set_option('future.no_silent_downcasting', True)
except Exception:
    pass

print("🚀 Starting Phase T.4: Chronological Predictability Evaluation (Trust-Aware)")

# ---------------------------------------------------------
# DEPENDENCY GUARD & PATHS
# ---------------------------------------------------------
required_globals = ["TARGET_REPO", "BASE_DIR", "TIME_SPLIT_RATIO"]
for var in required_globals:
    if var not in globals() or globals()[var] is None:
        raise RuntimeError(f"❌ Phase T.4 Dependency Error: '{var}' is missing. Did you run Phase 0 and T.2?")

base_dir_p = Path(BASE_DIR)
TARGET_REPO = str(TARGET_REPO)
TIME_SPLIT_RATIO = float(TIME_SPLIT_RATIO)

# T.4 STRICT LOCK: Bind directly to the established Temporal pipeline tracks
TEST_CSV_PATH = base_dir_p / f"t2_test_chrono_{int(TIME_SPLIT_RATIO*100)}_{TARGET_REPO}.csv"
TRAIN_CSV_PATH = base_dir_p / f"t2_train_chrono_{int(TIME_SPLIT_RATIO*100)}_{TARGET_REPO}.csv"
CHRONO_THRESHOLDS_PATH = base_dir_p / f"t3_inflection_thresholds_{TARGET_REPO}_chrono_locked.json"
EVAL_CSV_PATH = base_dir_p / f"t4_evaluation_matrix_chrono_{TARGET_REPO}.csv"

for p in [TEST_CSV_PATH, TRAIN_CSV_PATH, CHRONO_THRESHOLDS_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"❌ Phase T.4 Error: Required file not found: {p.name}")

df_test = pd.read_csv(TEST_CSV_PATH)
df_train = pd.read_csv(TRAIN_CSV_PATH)

# Required-column guards (Ported from B.4/E.4)
required_cols = {"commit_sha", "file_basename", "method_name", "granular_rule", "metric_value", "is_refactored", "Calculated_MethodCount"}
missing_test = required_cols - set(df_test.columns)
missing_train = required_cols - set(df_train.columns)

if missing_test:
    raise RuntimeError(f"❌ Phase T.4 Error: Test CSV missing required columns: {sorted(missing_test)}")
if missing_train:
    raise RuntimeError(f"❌ Phase T.4 Error: Train CSV missing required columns: {sorted(missing_train)}")

with open(CHRONO_THRESHOLDS_PATH, "r", encoding="utf-8") as f:
    locked_thresholds = json.load(f)

pmd_defaults = {
    "NcssCount_Class": 1500, "NcssCount_Method": 60,
    "CyclomaticComplexity_Class": 80, "CyclomaticComplexity_Method": 10,
    "ExcessivePublicCount": 45, "CouplingBetweenObjects": 20,
    "ExcessiveImports": 30, "TooManyMethods": 10,
    "ExcessiveParameterList": 10
}

# 3. Method count for TooManyMethods comes from D.2 (Calculated_MethodCount),
#    carried through the split CSVs — do NOT recompute it here. It must match
#    T.3's calibration metric exactly.

print(f"📊 Final Chronological Exam Loaded: Evaluating against Future Timeline.")

# 4. The Blind Evaluation Engine
f1_records = []

CALIBRATION_EXCLUDED_RULES = {"ExcessiveParameterList"}

rules_to_evaluate = [
    k for k in locked_thresholds.keys()
    if not k.startswith("_") and k not in CALIBRATION_EXCLUDED_RULES
]

for rule in rules_to_evaluate:
    if rule == "TooManyMethods":
        # MUST match T.3's calibration selection: one row per file, God-Class
        # label only, engineered method count as the metric.
        GOD_CLASS_RULES = ["NcssCount_Class", "CyclomaticComplexity_Class",
                           "ExcessivePublicCount", "TooManyMethods"]
        t_data = (df_test[df_test["granular_rule"].isin(GOD_CLASS_RULES)]
                  .drop_duplicates(subset=["commit_sha", "file_basename"]).copy())
        t_data["val"] = t_data["Calculated_MethodCount"]
        train_val = (df_train[df_train["granular_rule"].isin(GOD_CLASS_RULES)]
                     .drop_duplicates(subset=["commit_sha", "file_basename"])["Calculated_MethodCount"])
    else:
        t_data = df_test[df_test["granular_rule"] == rule].copy()
        t_data["val"] = t_data["metric_value"]
        train_val = df_train[df_train["granular_rule"] == rule]["metric_value"]

    if t_data.empty: continue
    t_actual = t_data["is_refactored"].astype(int)
    if t_actual.sum() == 0: continue

    rule_data_f1 = locked_thresholds.get(rule, {})
    rule_data_f05 = locked_thresholds.get("_precision_optimized_f0_5", {}).get(rule, {})

    trust_f1 = rule_data_f1.get("trustworthy", False)
    trust_f05 = rule_data_f05.get("trustworthy", False)

    raw_thresh_f1 = rule_data_f1.get("threshold_value")
    raw_thresh_f05 = rule_data_f05.get("threshold_value")

    pmd_thresh = pmd_defaults.get(rule, 9999)
    naive_thresh = int(math.ceil(train_val.median())) if not train_val.empty else 9999

    eval_thresh_f1 = raw_thresh_f1 if (trust_f1 and raw_thresh_f1 is not None) else pmd_thresh
    eval_thresh_f05 = raw_thresh_f05 if (trust_f05 and raw_thresh_f05 is not None) else pmd_thresh

    pmd_p, pmd_r, pmd_f1, pmd_f05, pmd_mcc, _, support = get_stats(t_actual, (t_data["val"] >= pmd_thresh), t_data["val"])
    naive_p, naive_r, naive_f1, naive_f05, naive_mcc, _, _ = get_stats(t_actual, (t_data["val"] >= naive_thresh), t_data["val"])
    c_f1_p, c_f1_r, c_f1_f1, c_f1_f05, c_f1_mcc, auc_pr, _ = get_stats(t_actual, (t_data["val"] >= eval_thresh_f1), t_data["val"])
    c_f05_p, c_f05_r, c_f05_f1, c_f05_f05, c_f05_mcc, _, _ = get_stats(t_actual, (t_data["val"] >= eval_thresh_f05), t_data["val"])

    f1_records.append({
        "Metric": rule, "Support": support,
        "PMD_Thresh": pmd_thresh,
        "Trust_F1": trust_f1, "Thresh_F1": eval_thresh_f1,
        "Trust_F05": trust_f05, "Thresh_F05": eval_thresh_f05,
        "Naive_Thresh": naive_thresh,
        "Hist_F1": rule_data_f1.get("kfold_avg_f1", 0.0) or 0.0,
        "Hist_F05": rule_data_f05.get("kfold_avg_f0_5", 0.0) or 0.0,
        "Cust_F1_F1": c_f1_f1, "Cust_F05_F05": c_f05_f05,
        "PMD_F1": pmd_f1, "PMD_F05": pmd_f05,
        "Naive_F1": naive_f1, "Naive_F05": naive_f05
    })

df_final = pd.DataFrame(f1_records)
if not df_final.empty:
    def evaluate_drift_f1(row):
        if not row["Trust_F1"]: return "➖ Uncalibrated Noise"
        cust, pmd, hist = float(row["Cust_F1_F1"]), float(row["PMD_F1"]), float(row["Hist_F1"])
        if cust < pmd: return "🚨 Severe (PMD Wins)"
        elif hist > 0 and (hist - cust) >= 0.05: return "⚠️ Degradation"
        return "✅ Stable"

    def evaluate_drift_f05(row):
        if not row["Trust_F05"]: return "➖ Uncalibrated Noise"
        cust, pmd, hist = float(row["Cust_F05_F05"]), float(row["PMD_F05"]), float(row["Hist_F05"])
        if cust < pmd: return "🚨 Severe (PMD Wins)"
        elif hist > 0 and (hist - cust) >= 0.05: return "⚠️ Degradation"
        return "✅ Stable"

    df_final["Drift_Alert_F1"] = df_final.apply(evaluate_drift_f1, axis=1)
    df_final["Drift_Alert_F05"] = df_final.apply(evaluate_drift_f05, axis=1)

    df_display = df_final.copy()
    df_display["Display_Thresh_F1"] = df_display.apply(lambda r: f"{r['Thresh_F1']}" if r["Trust_F1"] else f"⚠️ {r['Thresh_F1']} (Fallback)", axis=1)
    df_display["Display_Thresh_F05"] = df_display.apply(lambda r: f"{r['Thresh_F05']}" if r["Trust_F05"] else f"⚠️ {r['Thresh_F05']} (Fallback)", axis=1)

    float_cols = ["Hist_F1", "Hist_F05", "Cust_F1_F1", "Cust_F05_F05", "PMD_F1", "PMD_F05", "Naive_F1", "Naive_F05"]
    for c in float_cols: df_display[c] = df_display[c].apply(lambda x: f"{float(x):.3f}")

    df_f1 = df_display[["Metric", "Support", "PMD_Thresh", "Naive_Thresh", "Display_Thresh_F1", "Hist_F1", "Cust_F1_F1", "PMD_F1", "Naive_F1", "Drift_Alert_F1"]].copy()
    df_f1.rename(columns={"Display_Thresh_F1": "Cust_Thresh", "Cust_F1_F1": "Cust_F1"}, inplace=True)

    df_f05 = df_display[["Metric", "Support", "PMD_Thresh", "Naive_Thresh", "Display_Thresh_F05", "Hist_F05", "Cust_F05_F05", "PMD_F05", "Naive_F05", "Drift_Alert_F05"]].copy()
    df_f05.rename(columns={"Display_Thresh_F05": "Cust_Thresh", "Cust_F05_F05": "Cust_F05"}, inplace=True)

    print("\n📊 TABLE 1: F1-Optimized Chronological Evaluation")
    print(df_f1.to_markdown(index=False))

    print("\n🎯 TABLE 2: Precision-Optimized (F0.5) Chronological Evaluation")
    print(df_f05.to_markdown(index=False))

    trust_df_f1 = df_final[df_final["Trust_F1"] == True]
    trust_df_f05 = df_final[df_final["Trust_F05"] == True]

    print("\n" + "=" * 65)
    print("🏆 CHRONOLOGICAL EVALUATION SUMMARY (Trustworthy Rules Only)")
    print("=" * 65)

    if len(trust_df_f1) > 0:
        avg_pmd_f1 = (trust_df_f1["PMD_F1"] * trust_df_f1["Support"]).sum() / trust_df_f1["Support"].sum()
        avg_cust_f1 = (trust_df_f1["Cust_F1_F1"] * trust_df_f1["Support"]).sum() / trust_df_f1["Support"].sum()
        print("   [ TRACK 1: Standard F1 Score ]")
        print(f"   Rules Survived Time-Split:       {len(trust_df_f1)}")
        print(f"   Weighted PMD:                    {avg_pmd_f1:.3f}")
        print(f"   Weighted Custom:                 {avg_cust_f1:.3f}")
    else:
        print("   [ TRACK 1: Standard F1 Score ]")
        print("   ⚠️ No rules met the Trustworthy criteria.")

    print("-" * 65)

    if len(trust_df_f05) > 0:
        avg_pmd_f05 = (trust_df_f05["PMD_F05"] * trust_df_f05["Support"]).sum() / trust_df_f05["Support"].sum()
        avg_cust_f05 = (trust_df_f05["Cust_F05_F05"] * trust_df_f05["Support"]).sum() / trust_df_f05["Support"].sum()
        print("   [ TRACK 2: Precision-Weighted F0.5 Score ]")
        print(f"   Rules Survived Time-Split:       {len(trust_df_f05)}")
        print(f"   Weighted PMD:                    {avg_pmd_f05:.3f}")
        print(f"   Weighted Custom:                 {avg_cust_f05:.3f}")
    else:
        print("   [ TRACK 2: Precision-Weighted F0.5 Score ]")
        print("   ⚠️ No rules met the Trustworthy criteria.")
    print("=" * 65)

    df_final.to_csv(EVAL_CSV_PATH, index=False)
    print(f"\n💾 Saved chronological evaluation matrix to: {EVAL_CSV_PATH.name}")
else:
    print("⚠️ No valid evaluation records generated. Ensure the Test Set has positive signals.")

print("🏁 Phase T.4 Complete. Chronological evaluation finalized.")

## ⚙️ Phase 5: Master Cross-Regime Discrimination Diagnostic
**Objective:** Prove the underlying predictive limits of single metrics via Area Under the Precision-Recall Curve (AUC-PR).
**Logic Flow:**
* **Global Extraction:** Systematically iterate through every generated CSV matrix across the entire pipeline (Pooled, Spatial, Scarcity, Chronological, Train, and Test).
* **Threshold-Agnostic Scoring:** For every target rule, calculate the absolute AUC-PR directly on the raw metric values, bypassing discrete integer thresholds entirely.
* **Lift Calculation:** Divide the resulting AUC-PR by the dataset's base refactoring rate to calculate the statistical "Lift" (where a Lift of ~1.0 equals random chance).
* **Reliability Guarding:** Apply prevalence and positive-support guards to flag and exclude statistically noisy or unstable distributions.
* **Master Table:** Aggregate the results into a definitive summary table, mathematically proving whether the metrics possess the underlying discriminatory power to predict refactorings, independent of how they are thresholded.

In [ ]:
# =====================================================================
# DISCRIMINATION DIAGNOSTIC — across all sampling regimes
# Does ANY single PMD metric rank soon-to-be-refactored entities above
# the rest? AUC-PR for an uninformative metric equals the base rate, so
# lift = AUC-PR / base_rate ~ 1.0 means ranking at chance. We report n_pos
# and a reliability flag because lift is unstable when positives are few
# or prevalence is tiny — the headline is taken over well-supported rules.
# Note: AUC-PR evaluates the RAW metric scores, proving the underlying
# metric ranking power independent of any specific threshold.
# =====================================================================
import pandas as pd
from pathlib import Path
from sklearn.metrics import average_precision_score

print("\n" + "="*85)
print("🚀 INITIATING PHASE 6: MASTER CROSS-REGIME DISCRIMINATION DIAGNOSTIC")
print("="*85)

required_globals = ["TARGET_REPO", "BASE_DIR", "TIME_SPLIT_RATIO"]
for var in required_globals:
    if var not in globals() or globals()[var] is None:
        raise RuntimeError(f"❌ Phase 6 Dependency Error: '{var}' is missing.")

BASE_DIR_PATH    = Path(BASE_DIR)
TARGET_REPO      = str(TARGET_REPO)
TIME_SPLIT_RATIO = float(TIME_SPLIT_RATIO)

# Statistical confidence guards (Not arbitrary performance bounds)
MIN_POS_RELIABLE = 30      # below this many positives, statistical lift is highly noisy
MIN_PREVALENCE   = 0.05    # below this base rate, lift metrics can inflate due to severe imbalance

GOD_CLASS_RULES = ["NcssCount_Class", "CyclomaticComplexity_Class",
                   "ExcessivePublicCount", "TooManyMethods"]
RULES = ["NcssCount_Class", "CyclomaticComplexity_Class", "ExcessivePublicCount",
         "CouplingBetweenObjects", "ExcessiveImports", "NcssCount_Method",
         "CyclomaticComplexity_Method", "TooManyMethods"]

pct = int(TIME_SPLIT_RATIO * 100)

# Synchronized strictly to the outputs from our pipeline
SOURCES = [
    ("Pooled",           "all",     BASE_DIR_PATH / f"pmd_flat_{TARGET_REPO}.csv"),
    ("Spatial",          "train",   BASE_DIR_PATH / f"b2_train_spatial_80_{TARGET_REPO}.csv"),
    ("Scarcity",         "train",   BASE_DIR_PATH / f"e2_train_scarcity_{TARGET_REPO}.csv"),
    ("Chronological",    "train",   BASE_DIR_PATH / f"t2_train_chrono_{pct}_{TARGET_REPO}.csv"),
    ("Spatial/Scarcity", "holdout", BASE_DIR_PATH / f"b2_test_spatial_20_{TARGET_REPO}.csv"),
    ("Chronological",    "holdout", BASE_DIR_PATH / f"t2_test_chrono_{pct}_{TARGET_REPO}.csv"),
]

def metric_series(df, rule):
    # TooManyMethods is the synthetic, file-level metric: one row per file,
    # God-Class label, engineered method count — matching the calibration phases.
    if rule == "TooManyMethods":
        d = (df[df["granular_rule"].isin(GOD_CLASS_RULES)]
             .drop_duplicates(subset=["commit_sha", "file_basename"]))
        return d["Calculated_MethodCount"], d["is_refactored"].astype(int)
    d = df[df["granular_rule"] == rule]
    return d["metric_value"], d["is_refactored"].astype(int)

def discrimination(values, labels):
    n, n_pos = len(labels), int(labels.sum())
    if n == 0 or n_pos == 0 or n_pos == n:
        return {"n": n, "n_pos": n_pos, "base_rate": None, "auc_pr": None,
                "lift": None, "flag": "no positives"}
    base = labels.mean()
    ap = average_precision_score(labels, values)
    flag = "low n_pos" if n_pos < MIN_POS_RELIABLE else ("rare (<5%)" if base < MIN_PREVALENCE else "ok")
    return {"n": n, "n_pos": n_pos, "base_rate": round(float(base), 4),
            "auc_pr": round(float(ap), 4), "lift": round(float(ap / base), 2), "flag": flag}

rows, loaded = [], {}
for experiment, split, path in SOURCES:
    if not Path(path).exists():
        print(f"⚠️  Skipping {experiment}/{split}: {Path(path).name} not found.")
        continue
    df = pd.read_csv(path)
    need = {"granular_rule", "is_refactored", "commit_sha", "file_basename",
            "metric_value", "Calculated_MethodCount"}
    missing = need - set(df.columns)
    if missing:
        print(f"⚠️  Skipping {experiment}/{split}: missing {sorted(missing)} (re-run D.2).")
        continue
    loaded[(experiment, split)] = df
    for rule in RULES:
        vals, labs = metric_series(df, rule)
        rows.append({"Experiment": experiment, "Split": split, "Rule": rule, **discrimination(vals, labs)})

res = pd.DataFrame(rows)
res["Source"] = res["Experiment"] + "/" + res["Split"]

print("\n================ DISCRIMINATION BY SOURCE × RULE ================")
print("AUC-PR ~ base_rate  ⇒  lift ~ 1.0  ⇒  metric ranks refactoring at chance.\n")
print(res[["Source","Rule","n","n_pos","base_rate","auc_pr","lift","flag"]].to_markdown(index=False))

order = ["Pooled/all","Spatial/train","Scarcity/train","Chronological/train",
         "Spatial/Scarcity/holdout","Chronological/holdout"]
piv_lift = res.pivot(index="Rule", columns="Source", values="lift")
piv_auc  = res.pivot(index="Rule", columns="Source", values="auc_pr")
cols = [c for c in order if c in piv_lift.columns]

print("\n--- LIFT (AUC-PR / base_rate) by rule × source ---")
print("Lift strictly measures how much better the metric ranks over blind guessing.")
print(piv_lift[cols].to_markdown())

print("\n--- AUC-PR (absolute) by rule × source ---")
print(piv_auc[cols].to_markdown())

print("\n================ PER-SOURCE SUMMARY ================")
print(f"(reliable = n_pos≥{MIN_POS_RELIABLE} AND base_rate≥{MIN_PREVALENCE})\n")
summary = []
for (experiment, split), _ in loaded.items():
    block = res[(res.Experiment==experiment)&(res.Split==split)].dropna(subset=["auc_pr"])
    if block.empty: continue
    top = block.loc[block["auc_pr"].idxmax()]
    rel = block[block["flag"]=="ok"]
    lift_str = (f'{rel.loc[rel["lift"].idxmax()]["lift"]:.2f} ({rel.loc[rel["lift"].idxmax()]["Rule"]})'
                if not rel.empty else "— (no statistically stable rule)")
    summary.append({
        "Source": f"{experiment}/{split}",
        "Max Absolute AUC-PR": f'{top["auc_pr"]:.3f} ({top["Rule"]})',
        "Max Reliable Lift": lift_str
    })
print(pd.DataFrame(summary).to_markdown(index=False))

out_csv = BASE_DIR_PATH / f"discrimination_diagnostic_{TARGET_REPO}.csv"
res.drop(columns=["Source"]).to_csv(out_csv, index=False)
print(f"\n💾 Saved full diagnostic to: {out_csv.name}")
print("🏁 Phase 6 Complete. The overarching data structure has been successfully diagnosed.")